This notebook is the write-up plus runnable code for the **0.69536** submission. The first half explains the algorithm, metrics, features, and formulas. The second half is the full pipeline (Polars lazy + triple GBDT + dual-engine evidence).

| Public | Meaning |
|---|---|
| 0.65682 | Baseline |
| 0.6837 | notebook rate-manifold / partner-field / triple GBDT |
| 0.68507 | added `iso_preflop`, `late_vs_blind`, `dump_from_oop`, `partner_calls_dump` + evidence z-blend |
| **0.69536** | SPR / `amount_to` / board **evidence-only**; drop z-blend; score **every** eval pair; add rolling-20 + `isolation_purity_rate` to pair X |


**Contents**

1. Problem, PU labels, and metric
2. Algorithm — two feature spaces
3. Workflow
4. Feature list
5. Hand-level formulas
6. Pair-level formulas
7. Experiments not to repeat
8. Runnable pipeline


## 1. Problem, PU labels, and metric

The dataset is **synthetic 6-max NLHE**: ~2M hands, ~12k players, **400 pools × 30** with no mixing (`table_id`). Time split 60/40 by `hands.phase` (`development` / `evaluation`).

Each submission row is a **player pair** on eval (112,540 pairs):

| Column | Role | Weight |
|---|---|---|
| `risk_score` $\in [0,1]$ | collusion or not | 70% Pair AP |
| `predicted_behavior` | `none` or one of 3 published families | 10% Behavior MAP |
| `evidence_hand_1..5` | hands that include **both** players, or `NO_EVIDENCE` | 20% MAP@5 |

$$
\mathrm{Score} = 0.70 \cdot \mathrm{PairAP} + 0.20 \cdot \mathrm{MAP@5} + 0.10 \cdot \mathrm{BehaviorMAP}
$$

`other_coordination` is valid on the form but **does not enter Behavior MAP** — never predict that class.

### Labels are PU, not PN

`development_labels.csv`: 1,860 confirmed pairs (372 positive / 1,488 confirmed-negative). Every co-table pair **not listed is unlabeled, not negative**. Training only on the 1,860 labeled pairs yields OOF AP ~0.97 while public Pair AP collapses: the model never sees the “normal” pairs that dominate the 112k eval set.

Train matrix ~25,860 pairs = labeled + up to 60 unlabeled / table (`UNKNOWN_PER_TABLE`), excluding players already positive. Track **PU-stress AP** (treat unlabeled as negative, reweight $n_{\mathrm{eval}} / n_{\mathrm{PU}}$); do not pick models with labeled AP.

### MAP@5 (evidence)

For pair $p$, $R_p$ is the planted-hand set. The model returns an order $h_1,\ldots,h_5$. Let $\mathrm{hits}(k)$ be the number of relevant items in the first $k$ positions **after** $h_k$ is a hit:

$$
\mathrm{AP@5}(p)=\frac{1}{\min(|R_p|,5)}\sum_{k=1}^{5}\mathbf{1}[h_k\in R_p]\cdot\frac{\mathrm{hits}(k)}{k}
$$

Official: planted evidence is **specific behavior visible in `actions.parquet`**. Hidden chip flow alone is never evidence. Coordination is episodic — so evidence is **ranking hands within a pair**, not classifying the pair.

### Behavior MAP

Only pairs marked collusive (`risk` in the top rate) carry a family. Macro OvR AP over 3 classes. Isolation is hardest (OOF class AP was once ~0.49), so the isolation detector gets **sample weight 10**.


## 2. Algorithm — two outputs, two feature spaces

The system splits **pair risk** from **hand evidence**. On the same hand row, some columns may enter the ranker only.

```
actions / seats / hands
        │
        ▼
 pair × hand features  ──┬── aggregate ──► pair X ──► risk + family
                         │
                         └── within-pair ranks ──► evidence MAP@5
```

**Channel-split principle** (this is what won 0.68507 → 0.69536):

1. **Pair X** describes the pair's *long-run habits*: rates, imbalance, burst, partner-vs-field, 20-hand windows.
2. **Evidence X** describes *which hand in the pair is the episode*: SPR, `amount_to`, board texture, within-pair percentile, heuristic family priority.
3. Putting SPR/board into pair-X `mean/max/p95/top5` **hurt** the score (same pattern as HU check-check → 0.68434). Evidence-only columns live in `EVIDENCE_ONLY_COLS` and are dropped from pair aggregation.

### 2.1 Pair risk

Three independent GBDTs on the same pair X, fixed blend:

$$
\hat r = 0.40\,p_{\mathrm{XGB}} + 0.35\,p_{\mathrm{LGB}} + 0.25\,p_{\mathrm{Cat}}
$$

Three LightGBM one-vs-rest detectors (directed / soft / isolation). Family = $\arg\max$ of the three detectors. Isolation weight $10$ because class AP is low.

Positive rate $\rho$ is chosen on the grid $\{0.50\%,\ldots,1.40\%\}$: take the **smallest rate** still within $0.005$ of the best PU-stress Behavior MAP. Pairs outside the top-$\rho$ get `none`.

Sample weight at fit:

$$
w_i=\begin{cases}
2.0 & y_i=1\\
1.0 & \text{confirmed negative}\\
0.35 & \text{PU unlabeled}
\end{cases}
$$

CV: `StratifiedGroupKFold` by `table_id` — the 30 players in a pool must not leak into another fold.

### 2.2 Evidence

Global: HistGradientBoosting on all positive-pair hands.

By family: HGB + LightGBM + XGB + **XGBRanker** (listwise within pair):

$$
p_{\mathrm{quad}}=0.80\cdot\frac{p_{\mathrm{HGB}}+p_{\mathrm{LGB}}+p_{\mathrm{XGB}}}{3}+0.20\cdot s_{\mathrm{Ranker}}
$$

Final blend is **not z-scored** (winning ablation):

$$
s_{\mathrm{ev}}=0.60\,p_{\mathrm{spec}}+0.40\,p_{\mathrm{glob}}
$$

`p_spec` is chosen by the pair's **predicted family** (not the label — eval has no labels). Score **every** 112,540 pairs, in chunks of 8,000 `pair_id`, so we never `collect()` ~9.6M rows at once. Rank features are `over("pair_id")`, so chunking is valid.

Heuristic family-priority remains a `coalesce` fallback if the model misses a pair.


## 3. Workflow

```mermaid
flowchart TD
    A[Raw parquet + labels] --> B[Pair index + PU sample]
    B --> C[Action context: last_aggressor, SPR, amount_to]
    C --> D[Player × hand stats]
    D --> E[Pair × hand features]
    E --> F[Pair aggregate — drop EVIDENCE_ONLY]
    E --> G[Evidence matrix — SPR/board + pair percentile]
    F --> H[Partner vs field contrast]
    H --> I[5-fold grouped CV]
    I --> J["Ensemble risk 0.40 / 0.35 / 0.25"]
    I --> K[Family detectors + rate grid]
    G --> L["Evidence 0.60 spec + 0.40 global"]
    J --> M[submission.csv]
    K --> M
    L --> M
```

1. **Pair index**: every $\binom{6}{2}$ co-table pair per hand; join labels; PU sample.
2. **Action context**: `is_aggressive`, `last_aggressor` forward-fill by street, `amount_bb`, SPR, `amount_to_bb`.
3. **Player-hand**: contribution / net / hole strength / action counts.
4. **Pair-hand**: transfer, true HU (both still active), pressure on partner vs outsider, 3-bet templates, SPR/board (evidence-only).
5. **Aggregate**: mean / max / p95 / top-5 / rate above quantile 0.95; rates; 8-bin phase burst; rolling 20; table percentile.
6. **Partner-field**: behavior when sitting with the partner vs when the partner is absent.
7. **Fit** pair models → **fit** evidence models on labeled positives.
8. **Infer** eval risk/family, then evidence by chunk, write `/kaggle/working/submission.csv`.


## 4. Feature list

The base grain is **one row = one `(pair_id, hand_id)`**. Pair X is statistics over those rows, excluding the evidence-only group.

### 4.1 Player × hand (before pairing)

| Group | Column | Meaning |
|---|---|---|
| Stack / pot | `stack_bb`, `contribution_bb`, `net_bb`, `pot_bb` | normalized by BB |
| Outcome | `folded`, `went_to_showdown`, `won_share` | hand outcome |
| Hole | `hole_strength` | high card + pair + suited (formula below) |
| Action counts | `n_actions`, `aggressive_actions`, `raises`, `calls`, `checks`, `folds`, `all_ins`, `postflop_checks`, `folds_facing_bet` | frequencies |
| Size | `max_amount_bb`, `max_to_call_bb`, `max_amount_pot_ratio` | action size |

### 4.2 Pair × hand — chip flow & showdown

| Column | Meaning |
|---|---|
| `transfer_1_to_2_bb`, `transfer_2_to_1_bb`, `transfer_any_bb` | chips flowing between the two players in the hand |
| `net_gap_bb`, `contrib_gap_bb`, `flow_oneway` | asymmetry |
| `dump_vs_hole` | dump × donor hole strength / 14 |
| `both_showdown`, `one_folded`, `both_folded` | synchronized outcome |
| `pair_pot_share`, `transfer_pot_ratio` | pot-normalized ratios |
| `passivity_density` | (HU check + call) / HU actions |
| `is_pure_dump` | transfer $\ge 18$ BB (one side folded) |
| `is_soft_checkdown` | both show down and there is a HU check on turn/river |
| `is_squeeze_isolation` | pair raise + outsider fold |

### 4.3 Pair × hand — sequence / seat (won 0.68507)

| Column | Template |
|---|---|
| `pair_open_3bet` | one player opens, the other 3-bets preflop |
| `n_out_fold_after_3bet` | outsider folds after the 3-bet |
| `iso_preflop` | open→3bet **and** outsider fold (no check-down required) |
| `late_vs_blind` | one player late (CO/BTN), one player blind |
| `dump_from_oop` | donor is in the blinds |
| `partner_calls_dump` | there is a transfer and the partner calls when facing a bet |

### 4.4 Evidence-only — **do not** `mean/max/p95/top5` these on pair X

| Column | Source |
|---|---|
| `spr_min`, `spr_med` | $\mathrm{stack\_before}/\max(\mathrm{pot\_before},BB)$ of a member |
| `dump_spr` | minimum SPR of the **donor** |
| `max_amount_to_bb`, `max_aggr_amount_to_bb` | `amount_to` / BB |
| `board_paired`, `board_monotone`, `board_twotone`, `n_board_cards` | flop/board texture |

They enter the evidence ranker only (+ percentile / to-max within the pair).

### 4.5 Pair X — long-run statistics

From each hand col $c$ (not evidence-only): $c_{\mathrm{mean}}$, $c_{\mathrm{max}}$, $c_{\mathrm{p95}}$, and for the tail: $c_{\mathrm{top5}}$, $c_{\mathrm{rate95}}$.

Rates / imbalance:

| Column | Meaning |
|---|---|
| `transfer_imbalance` | $\lvert C_{12}-C_{21}\rvert / (C_{12}+C_{21})$ |
| `transfer_per_hand`, `net_flow_ratio` | average flow |
| `isolation_imbalance` | outsider folds vs partner folds |
| `isolation_purity_rate` | $\mathbb{E}[\mathbf{1}\{F_{\mathrm{out}}\ge 1 \land F_{\mathrm{partner}}=0\}]$ |
| `pure_dump_rate`, `soft_checkdown_rate`, `squeeze_isolation_rate` | template frequencies |
| `outsider_fold_efficiency`, `partner_immunity_ratio` | fold/call mix |
| `transfer_roll20_max`, `isolation_roll20_max` | max 20-hand window sum by `phase_progress` |
| `directed_burst_max`, `soft_burst_max`, `isolation_burst_max` | max mean signal over 8 phase bins |
| `transfer_top5_share` | chip dumps concentrated in a few hands |

Each metric in `TABLE_RANK_METRICS` also gets `_table_pct` = rank at the table / number of table pairs.

### 4.6 Partner vs field

For player $i$ in the pair, compare expected behavior **when the partner is at the table** vs **when the partner is absent** (same `table_id`): contrast, shrink, $t$-stat, ratio. Then add/subtract the two sides → `pair_net_contrast`, `directed_asymmetry`, `soft_contrast`, …

### 4.7 Evidence extras (not pair X)

For each hand col $c$: $c_{\mathrm{pair\_pct}}=\mathrm{rank}_p(c)/n_p$ and $c_{\mathrm{to\_max}}=c / \max_p(c)$. The ranker sees “this hand is unusual **within this pair**”.


## 5. Hand-level formulas

Notation: $n_1,n_2$ = `net_bb`; $BB$ = big blind. Seat position modulo 6 from UTG.

### Hole strength

$$
s=\max(r_1,r_2)+7\cdot\mathbf{1}[r_1=r_2]+1.5\cdot\mathbf{1}[\text{suited}]
$$

$r\in\{2,\ldots,14\}$ (T=10, …, A=14).

### Chip transfer in one hand

$$
T_{1\to 2}=\min\bigl(\max(-n_1,0),\ \max(n_2,0)\bigr),\qquad
T_{2\to 1}=\min\bigl(\max(-n_2,0),\ \max(n_1,0)\bigr)
$$

$$
T=\max(T_{1\to 2},T_{2\to 1})
$$

This is the chips **one player lost and the other received**, not the absolute gap.

### SPR and bet-to

$$
\mathrm{SPR}=\mathrm{clip}\!\left(\frac{\texttt{stack\_before}}{\max(\texttt{pot\_before},BB)},\,0,\,200\right)
$$

$$
\texttt{amount\_to\_bb}=\texttt{amount\_to}/BB
$$

### True heads-up

A member's action is counted as HU only when `players_active=2` **and** the partner has not already folded before that action. This avoids a “fake HU” while a third player is still in the pot.

### Family signals (GBDT input + evidence fallback)

$$
\begin{aligned}
s_{\mathrm{dir}}&=\log\bigl(1+1.5T+0.8G+25D+15F_{\mathrm{flop}}+12C_{\mathrm{dump}}\bigr)+0.3\log(1+A_{\max})\\
s_{\mathrm{soft}}&=\log\bigl(1+40\cdot\mathrm{SCD}+25\cdot\mathrm{SD}+8C_{\mathrm{late}}+5C_{\mathrm{HU}}\bigr)+0.3\log(1+G)\\
s_{\mathrm{iso}}&=\log\bigl(1+30Q+22I_{\mathrm{pf}}+12F_{\mathrm{out}}+6R+8B_{3}+G_{\mathrm{contrib}}\bigr)
\end{aligned}
$$

Priority (no log; used as heuristic evidence):

$$
\begin{aligned}
\pi_{\mathrm{dir}}&=2.5T+1.2G+40D+25F_{\mathrm{flop}}+18C_{\mathrm{dump}}+0.3\,\mathrm{pot}\\
\pi_{\mathrm{soft}}&=60\cdot\mathrm{SCD}+40\cdot\mathrm{SD}+12C_{\mathrm{late}}+1.2G+0.3\,\mathrm{pot}\\
\pi_{\mathrm{iso}}&=45Q+35I_{\mathrm{pf}}+15F_{\mathrm{out}}+8R+2G_{\mathrm{contrib}}+8\cdot\mathrm{late\_vs\_blind}+0.4\,\mathrm{pot}
\end{aligned}
$$

`iso_preflop` $= \mathbf{1}[\texttt{pair\_open\_3bet}=1 \land \texttt{n\_out\_fold\_after\_3bet}\ge 1]$. Squeeze **does not** require a check-down — requiring a check-down used to hurt isolation class AP.


## 6. Pair-level formulas

$H_p$ = co-table hands of pair $p$. $N=|H_p|$.

### Isolation purity

$$
\texttt{isolation\_purity\_rate}(p)=\frac{1}{N}\sum_{h\in H_p}\mathbf{1}\bigl[F_{\mathrm{out}}(h)\ge 1 \land F_{\mathrm{partner}}(h)=0\bigr]
$$

This is a **hand rate**, not a new 3-bet template. Outsider folds while the partner does not = clean isolation.

### Rolling 20

Sort by `phase_progress`, causal window of length 20:

$$
W_T(t)=\sum_{j=0}^{19} T_{t-j},\qquad
\texttt{transfer\_roll20\_max}=\max_t W_T(t)
$$

Likewise $W_I(t)$ on `isolation_signal`. Catches clustered episodes, unlike burst over 8 even bins.

### Transfer imbalance

$$
\texttt{transfer\_imbalance}=\frac{\lvert\sum T_{1\to 2}-\sum T_{2\to 1}\rvert}{\sum T_{1\to 2}+\sum T_{2\to 1}+\varepsilon}
$$

### Partner-field shrink / $t$ / ratio

$\mu_{\mathrm{with}}$, $\mu_{\mathrm{field}}$ = mean behavior with / without the partner. $n_{\mathrm{with}}$ = number of hands with the partner.

$$
\begin{aligned}
\Delta&=\mu_{\mathrm{with}}-\mu_{\mathrm{field}}\\
\Delta_{\mathrm{shrunk}}&=\Delta\cdot\frac{n_{\mathrm{with}}}{n_{\mathrm{with}}+25}\\
\Delta_{t}&=\Delta\cdot\sqrt{n_{\mathrm{with}}+1}\\
\rho&=(\mu_{\mathrm{with}}+0.05)/(\mu_{\mathrm{field}}+0.05)
\end{aligned}
$$

The constant $25$ is empirical Bayes: noisy contrast on pairs with few hands is shrunk toward 0.

### Table percentile

$$
\texttt{x\_table\_pct}(p)=\frac{\mathrm{rank}_{\mathrm{table}}(x_p)}{n_{\mathrm{table}}+\varepsilon}
$$

The same 40 BB dump means different things at different stakes/tables.

### Evidence within-pair

$$
c_{\mathrm{pair\_pct}}=\frac{\mathrm{rank}_{p}(c)}{n_p},\qquad
c_{\mathrm{to\_max}}=\frac{c}{\max_{h\in H_p}c+\varepsilon}
$$


## 7. Experiments **not** to repeat

Same triple GBDT on the old A+B features: **0.651** (loses to 0.65682). Boosters do not rescue the wrong features.

Hurt public, do not retry:

- HU check-check / `ip_checks_river_hu` / `donor_stability` as pair rates (0.68507 → 0.68434)
- SPR/board/`amount_to` into pair `mean/max/p95` (reason for splitting `EVIDENCE_ONLY_COLS`)
- Evidence z-blend 0.85 model / 0.15 heuristic (dropped at 0.69536)
- Scoring only the top 50k pairs for evidence (dropped; now the full eval set)
- `dump_allin_weak` (the dumper is **not** weak)
- Squeeze requiring a postflop check-down
- `other_coordination` as a fourth class
- Raising the positive-rate cap above 1.4%
- Extra GBDT / TabPFN on the **same** pair X
- Clique, cross-phase history, table `XGBRanker` on `risk_score`

0.69536 is **not** expected to be a stable 0.70+; Evidence MAP is still the largest remaining room if the generator changes templates.


## 8. Runnable pipeline

The code below is the **exact codebase** of public **0.69536**. Runs on Kaggle CPU, internet off. Cache: `/kaggle/working/prepared_140926_s4` — delete that directory if you change the feature schema.

Cell order: constants → helpers → pair-hand → aggregate → evidence engines → `main()` → call `main()`.


### 8.1 Imports and constants


In [ ]:
from __future__ import annotations

import gc
import shutil
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm
from xgboost import XGBClassifier, XGBRanker
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

SEED = 42
UNKNOWN_PER_TABLE = 60
EVIDENCE_PAIR_CHUNK = 8_000
OUTPUT_DIR = Path("/kaggle/working")
PREP_DIR = OUTPUT_DIR / "prepared_140926_s4"
BEHAVIORS = ["directed_transfer", "soft_play", "coordinated_isolation"]
BEHAVIOR_TO_ID = {"none": 0, "directed_transfer": 1, "soft_play": 2, "coordinated_isolation": 3}
REQUIRED_FILES = {
    "players.parquet",
    "hands.parquet",
    "seats.parquet",
    "actions.parquet",
    "development_labels.csv",
    "development_evidence.csv",
    "evaluation_pairs.csv",
    "sample_submission.csv",
}
AGGRESSIVE = ["bet", "raise"]
ACTION_FEATURES = [
    "n_actions",
    "aggressive_actions",
    "raises",
    "calls",
    "checks",
    "folds",
    "all_ins",
    "postflop_checks",
    "folds_facing_bet",
    "max_amount_bb",
    "max_to_call_bb",
    "max_amount_pot_ratio",
]
PLAYER_COLS = [
    "seat_no",
    "pot_bb",
    "stack_bb",
    "contribution_bb",
    "net_bb",
    "folded",
    "went_to_showdown",
    "won_share",
    "hole_strength",
    *ACTION_FEATURES,
]
FIELD_COLS = [
    "net_bb",
    "aggressive_actions",
    "raises",
    "calls",
    "checks",
    "folds",
    "folds_facing_bet",
    "all_ins",
    "contribution_bb",
]
META_COLS = [
    "player_id",
    "account_age_days",
    "experience_hands_bucket",
    "preferred_stake",
    "region_bucket",
    "client_family",
]
BASE_PLAYER_COLS = ["player_id", "hands", "base_aggression", "base_showdown", "base_calls", "base_checks"]
TABLE_RANK_METRICS = [
    "transfer_imbalance",
    "transfer_per_hand",
    "net_flow_ratio",
    "showdown_check_ratio",
    "both_showdown_rate",
    "outsider_fold_rate",
    "isolation_imbalance",
    "contrib_asymmetry_ratio",
    "pure_dump_rate",
    "soft_checkdown_rate",
    "squeeze_isolation_rate",
    "transfer_any_bb_max",
    "net_gap_bb_max",
    "pot_bb_max",
    "outsider_fold_efficiency",
    "partner_immunity_ratio",
    "hu_postflop_check_ratio",
    "directed_burst_max",
    "soft_burst_max",
    "isolation_burst_max",
    "soft_burst_excess",
    "transfer_top5_share",
    "iso_preflop_mean",
    "late_vs_blind_mean",
    "isolation_purity_rate",
    "transfer_roll20_max",
    "isolation_roll20_max",
]
EVIDENCE_ONLY_COLS = {
    "spr_min",
    "spr_med",
    "max_amount_to_bb",
    "max_aggr_amount_to_bb",
    "dump_spr",
    "board_paired",
    "board_monotone",
    "board_twotone",
    "n_board_cards",
}


### 8.2 Helpers

Locate the dataset, parse rank/suit/board, SPR rolling-20, build the pair index + PU sample, action context (`last_aggressor`, SPR, `amount_to_bb`), player × hand.


In [ ]:
def find_data_dir() -> Path:
    roots = [Path("/kaggle/input"), Path("data"), Path(".")]
    found: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("players.parquet"):
            parent = path.parent
            names = {child.name for child in parent.iterdir()}
            if REQUIRED_FILES.issubset(names):
                found.append(parent)
    unique = list(dict.fromkeys(found))
    if unique:
        return unique[0]
    for fallback in (
        Path("/kaggle/input/detect-suspicious-value-transfers-in-poker"),
        Path("/kaggle/input/competitions/detect-suspicious-value-transfers-in-poker"),
    ):
        if fallback.exists():
            return fallback
    raise FileNotFoundError("Mount the competition dataset.")


def add_pair_key(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.min_horizontal("player_1", "player_2").alias("p_low"),
        pl.max_horizontal("player_1", "player_2").alias("p_high"),
    )


def card_rank(column: str) -> pl.Expr:
    rank = pl.col(column).cast(pl.Utf8).str.replace(r"[cdhsCDHS]$", "").str.to_uppercase()
    return (
        pl.when(rank == "A")
        .then(14)
        .when(rank == "K")
        .then(13)
        .when(rank == "Q")
        .then(12)
        .when(rank == "J")
        .then(11)
        .when(rank == "T")
        .then(10)
        .otherwise(rank.cast(pl.Int8, strict=False))
    )


def board_card_columns(names: list[str]) -> list[str]:
    skip = ("blind", "pot", "progress", "table", "phase", "start", "count", "id")
    found: list[str] = []
    for name in names:
        low = name.lower()
        if any(tok in low for tok in skip):
            continue
        if any(tok in low for tok in ("flop", "turn", "river", "board_card", "board_c", "community")):
            found.append(name)
        elif low.startswith("board") and low not in {"board_id"}:
            found.append(name)
    return list(dict.fromkeys(found))


def rolling_sum_pair(col: str, alias: str, window: int = 20) -> pl.Expr:
    try:
        expr = pl.col(col).rolling_sum(window_size=window, min_samples=1)
    except TypeError:
        expr = pl.col(col).rolling_sum(window_size=window, min_periods=1)
    return expr.over("pair_id").alias(alias)


def board_texture(hands: pl.LazyFrame) -> pl.LazyFrame:
    names = list(hands.collect_schema().names())
    cols = board_card_columns(names)
    keys = hands.select("hand_id").unique()
    if not cols:
        return keys.with_columns(
            pl.lit(0).cast(pl.Int8).alias("board_paired"),
            pl.lit(0).cast(pl.Int8).alias("board_monotone"),
            pl.lit(0).cast(pl.Int8).alias("board_twotone"),
            pl.lit(0).cast(pl.Int8).alias("n_board_cards"),
        )
    ranked = hands.select(["hand_id", *cols]).with_columns(
        [card_rank(c).alias(f"_br{i}") for i, c in enumerate(cols)]
        + [pl.col(c).cast(pl.Utf8).str.slice(-1, 1).str.to_lowercase().alias(f"_bs{i}") for i, c in enumerate(cols)]
    )
    rank_cols = [f"_br{i}" for i in range(len(cols))]
    suit_cols = [f"_bs{i}" for i in range(len(cols))]
    paired = pl.lit(False)
    for i, j in combinations(range(len(cols)), 2):
        paired = paired | (
            pl.col(rank_cols[i]).is_not_null()
            & pl.col(rank_cols[j]).is_not_null()
            & (pl.col(rank_cols[i]) == pl.col(rank_cols[j]))
        )
    n_cards = pl.sum_horizontal([pl.col(c).is_not_null().cast(pl.Int8) for c in rank_cols])
    flop_suits = suit_cols[: min(3, len(suit_cols))]
    monotone = pl.lit(False)
    if len(flop_suits) >= 3:
        monotone = (
            pl.col(flop_suits[0]).is_not_null()
            & (pl.col(flop_suits[0]) == pl.col(flop_suits[1]))
            & (pl.col(flop_suits[1]) == pl.col(flop_suits[2]))
        )
    twotone = pl.lit(False)
    if len(flop_suits) >= 3:
        twotone = (
            pl.col(flop_suits[0]).is_not_null()
            & pl.col(flop_suits[1]).is_not_null()
            & pl.col(flop_suits[2]).is_not_null()
            & ~monotone
            & (
                (pl.col(flop_suits[0]) == pl.col(flop_suits[1]))
                | (pl.col(flop_suits[0]) == pl.col(flop_suits[2]))
                | (pl.col(flop_suits[1]) == pl.col(flop_suits[2]))
            )
        )
    return ranked.with_columns(
        paired.cast(pl.Int8).alias("board_paired"),
        monotone.cast(pl.Int8).alias("board_monotone"),
        twotone.cast(pl.Int8).alias("board_twotone"),
        n_cards.cast(pl.Int8).alias("n_board_cards"),
    ).select(["hand_id", "board_paired", "board_monotone", "board_twotone", "n_board_cards"])


def behavior_id_expr() -> pl.Expr:
    return (
        pl.when(pl.col("behavior_family") == "directed_transfer")
        .then(1)
        .when(pl.col("behavior_family") == "soft_play")
        .then(2)
        .when(pl.col("behavior_family") == "coordinated_isolation")
        .then(3)
        .otherwise(-1)
        .cast(pl.Int8)
        .alias("behavior_id")
    )


def all_pair_hands(seats: pl.LazyFrame, hands: pl.LazyFrame, phase: str) -> pl.LazyFrame:
    hand_players = (
        seats.join(
            hands.filter(pl.col("phase") == phase).select(["hand_id", "table_id", "started_at"]),
            on="hand_id",
        )
        .group_by(["hand_id", "table_id", "started_at"])
        .agg(pl.col("player_id").sort_by("seat_no").alias("players"))
        .collect(engine="streaming")
        .sort(["table_id", "started_at", "hand_id"])
        .with_columns(
            (pl.col("started_at").rank("ordinal").over("table_id") / pl.len().over("table_id"))
            .cast(pl.Float32)
            .alias("phase_progress")
        )
    )
    frames = [
        hand_players.lazy().select(
            [
                "hand_id",
                "table_id",
                "phase_progress",
                pl.col("players").list.get(i).alias("a"),
                pl.col("players").list.get(j).alias("b"),
            ]
        )
        for i, j in combinations(range(6), 2)
    ]
    return (
        pl.concat(frames)
        .with_columns(
            pl.min_horizontal("a", "b").alias("p_low"),
            pl.max_horizontal("a", "b").alias("p_high"),
        )
        .drop(["a", "b"])
    )


def build_pair_indexes(
    seats: pl.LazyFrame,
    hands: pl.LazyFrame,
    dev_labels: pl.DataFrame,
    eval_pairs: pl.DataFrame,
    paths: dict[str, Path],
) -> tuple[pl.DataFrame, pl.DataFrame]:
    label_keys = add_pair_key(dev_labels)
    eval_keys = add_pair_key(eval_pairs)
    positive_players = set(
        dev_labels.filter(pl.col("label") == 1)
        .select(pl.concat_list("player_1", "player_2").alias("p"))
        .explode("p")["p"]
        .to_list()
    )
    print(f"Positive players excluded from PU sampling: {len(positive_players):,}")

    dev_all = all_pair_hands(seats, hands, "development")
    dev_counts = (
        dev_all.group_by(["p_low", "p_high"])
        .agg(pl.len().cast(pl.Int32).alias("shared_hands"), pl.col("table_id").first())
        .collect(engine="streaming")
        .join(
            label_keys.select(["pair_id", "p_low", "p_high", "label", "label_status", "behavior_family"]),
            on=["p_low", "p_high"],
            how="left",
        )
    )
    min_shared = int(eval_pairs["shared_hands"].min())
    known = dev_counts.filter(pl.col("pair_id").is_not_null()).with_columns(
        pl.lit(True).alias("is_labeled"),
        pl.lit(False).alias("is_pu"),
    )
    unknown = (
        dev_counts.filter(
            pl.col("pair_id").is_null()
            & (pl.col("shared_hands") >= min_shared)
            & ~pl.col("p_low").is_in(list(positive_players))
            & ~pl.col("p_high").is_in(list(positive_players))
        )
        .sample(fraction=1, shuffle=True, seed=SEED)
        .group_by("table_id", maintain_order=True)
        .head(UNKNOWN_PER_TABLE)
        .with_columns(
            pl.concat_str([pl.lit("U"), "p_low", "p_high"], separator="_").alias("pair_id"),
            pl.lit(False).alias("is_labeled"),
            pl.lit(True).alias("is_pu"),
        )
    )
    dev_pairs = (
        pl.concat([known, unknown], how="diagonal_relaxed")
        .with_columns(behavior_id_expr())
        .rename({"p_low": "player_1", "p_high": "player_2"})
        .sort(["table_id", "pair_id"])
    )
    dev_pair_hands = (
        dev_all.join(
            dev_pairs.lazy().select(
                ["pair_id", pl.col("player_1").alias("p_low"), pl.col("player_2").alias("p_high")]
            ),
            on=["p_low", "p_high"],
            how="inner",
        )
        .select(["pair_id", "hand_id", "table_id", "phase_progress"])
        .collect(engine="streaming")
    )
    dev_pairs.write_parquet(paths["dev_pairs"])
    dev_pair_hands.write_parquet(paths["dev_pair_hands"])
    del dev_all, dev_counts, known, unknown, dev_pair_hands
    gc.collect()

    eval_all = all_pair_hands(seats, hands, "evaluation")
    eval_pair_hands = (
        eval_all.join(
            eval_keys.lazy().select(["pair_id", "p_low", "p_high"]),
            on=["p_low", "p_high"],
            how="inner",
        )
        .select(["pair_id", "hand_id", "table_id", "phase_progress"])
        .collect(engine="streaming")
    )
    eval_tables = eval_pair_hands.group_by("pair_id").agg(
        pl.col("table_id").first(),
        pl.len().cast(pl.Int32).alias("shared_hands_calc"),
    )
    eval_pairs_prep = (
        eval_keys.select(
            ["pair_id", pl.col("p_low").alias("player_1"), pl.col("p_high").alias("player_2"), "shared_hands"]
        )
        .join(eval_tables, on="pair_id", how="left")
        .sort(["table_id", "pair_id"])
    )
    eval_pairs_prep.write_parquet(paths["eval_pairs"])
    eval_pair_hands.write_parquet(paths["eval_pair_hands"])
    del eval_all, eval_pair_hands
    gc.collect()
    return dev_pairs, eval_pairs_prep


def build_action_context(actions: pl.LazyFrame, hands: pl.LazyFrame, path: Path) -> None:
    action_names = set(actions.collect_schema().names())
    lf = (
        actions.join(hands.select(["hand_id", "big_blind"]), on="hand_id")
        .sort(["hand_id", "action_no"])
        .with_columns(
            (
                pl.col("action").is_in(AGGRESSIVE)
                | ((pl.col("action") == "all_in") & (pl.col("amount") > pl.col("to_call")))
            ).alias("is_aggressive")
        )
        .with_columns(
            pl.when(pl.col("is_aggressive")).then(pl.col("player_id")).otherwise(None).alias("_aggressor")
        )
        .with_columns(
            pl.col("_aggressor").shift(1).forward_fill().over(["hand_id", "street"]).alias("last_aggressor"),
            (pl.col("amount") / pl.col("big_blind")).cast(pl.Float32).alias("amount_bb"),
            (pl.col("to_call") / pl.col("big_blind")).cast(pl.Float32).alias("to_call_bb"),
            (pl.col("amount") / pl.max_horizontal("pot_before", "big_blind")).clip(0, 20).cast(pl.Float32).alias(
                "amount_pot_ratio"
            ),
        )
    )
    extra_expr = []
    if "stack_before" in action_names:
        extra_expr.append(
            (pl.col("stack_before") / pl.max_horizontal("pot_before", "big_blind"))
            .clip(0, 200)
            .cast(pl.Float32)
            .alias("spr")
        )
    if "amount_to" in action_names:
        extra_expr.append((pl.col("amount_to") / pl.col("big_blind")).cast(pl.Float32).alias("amount_to_bb"))
    if extra_expr:
        lf = lf.with_columns(extra_expr)
    computed = []
    if "stack_before" in action_names:
        computed.append("spr")
    if "amount_to" in action_names:
        computed.append("amount_to_bb")
    lf.select(
        [
            "hand_id",
            "action_no",
            "street",
            "player_id",
            "action",
            "to_call",
            "players_active",
            "is_aggressive",
            "last_aggressor",
            "amount_bb",
            "to_call_bb",
            "amount_pot_ratio",
            *computed,
        ]
    ).sink_parquet(path)


def build_player_hands(
    seats: pl.LazyFrame,
    hands: pl.LazyFrame,
    action_path: Path,
    player_path: Path,
    baseline_path: Path,
) -> None:
    action_by_player = (
        pl.scan_parquet(action_path)
        .group_by(["hand_id", "player_id"])
        .agg(
            pl.len().cast(pl.Int16).alias("n_actions"),
            pl.col("is_aggressive").sum().cast(pl.Int16).alias("aggressive_actions"),
            (pl.col("action") == "raise").sum().cast(pl.Int16).alias("raises"),
            (pl.col("action") == "call").sum().cast(pl.Int16).alias("calls"),
            (pl.col("action") == "check").sum().cast(pl.Int16).alias("checks"),
            (pl.col("action") == "fold").sum().cast(pl.Int16).alias("folds"),
            (pl.col("action") == "all_in").sum().cast(pl.Int16).alias("all_ins"),
            ((pl.col("street") != "preflop") & (pl.col("action") == "check"))
            .sum()
            .cast(pl.Int16)
            .alias("postflop_checks"),
            ((pl.col("action") == "fold") & (pl.col("to_call") > 0)).sum().cast(pl.Int16).alias("folds_facing_bet"),
            pl.col("amount_bb").max().cast(pl.Float32).alias("max_amount_bb"),
            pl.col("to_call_bb").max().cast(pl.Float32).alias("max_to_call_bb"),
            pl.col("amount_pot_ratio").max().cast(pl.Float32).alias("max_amount_pot_ratio"),
        )
    )
    (
        seats.join(hands.select(["hand_id", "phase", "big_blind", "final_pot"]), on="hand_id")
        .with_columns(
            card_rank("hole_card_1").alias("r1"),
            card_rank("hole_card_2").alias("r2"),
            (pl.col("starting_stack") / pl.col("big_blind")).cast(pl.Float32).alias("stack_bb"),
            (pl.col("total_contribution") / pl.col("big_blind")).cast(pl.Float32).alias("contribution_bb"),
            (pl.col("net_chips") / pl.col("big_blind")).cast(pl.Float32).alias("net_bb"),
            (pl.col("final_pot") / pl.col("big_blind")).cast(pl.Float32).alias("pot_bb"),
        )
        .with_columns(
            (
                pl.max_horizontal("r1", "r2")
                + 7 * (pl.col("r1") == pl.col("r2")).cast(pl.Int8)
                + 1.5
                * (pl.col("hole_card_1").str.slice(-1, 1) == pl.col("hole_card_2").str.slice(-1, 1)).cast(pl.Int8)
            )
            .cast(pl.Float32)
            .alias("hole_strength")
        )
        .join(action_by_player, on=["hand_id", "player_id"], how="left")
        .with_columns([pl.col(c).fill_null(0) for c in ACTION_FEATURES] + [pl.col("seat_no").cast(pl.Int16)])
        .select(["hand_id", "player_id", "phase", *PLAYER_COLS])
        .sink_parquet(player_path)
    )
    (
        pl.scan_parquet(player_path)
        .group_by(["phase", "player_id"])
        .agg(
            pl.len().alias("hands"),
            pl.col("contribution_bb").mean().alias("base_contribution"),
            pl.col("aggressive_actions").mean().alias("base_aggression"),
            pl.col("calls").mean().alias("base_calls"),
            pl.col("checks").mean().alias("base_checks"),
            pl.col("went_to_showdown").mean().alias("base_showdown"),
        )
        .sink_parquet(baseline_path)
    )


def player_side(player_path: Path, side: str, player_col: str) -> pl.LazyFrame:
    return pl.scan_parquet(player_path).select(
        "hand_id",
        pl.col("player_id").alias(player_col),
        *[pl.col(c).alias(f"{c}_{side}") for c in PLAYER_COLS],
    )


### 8.3 Pair × hand features

Transfer, true HU, partner/outsider pressure, 3-bet / dump / isolation templates, **and** SPR/board (dropped from pair X at aggregation).


In [ ]:
def build_pair_hand_features(
    pair_hands_path: Path,
    pairs_path: Path,
    player_path: Path,
    action_path: Path,
    out_path: Path,
    hands: pl.LazyFrame,
) -> None:
    pair_base = pl.scan_parquet(pair_hands_path).join(
        pl.scan_parquet(pairs_path).select(["pair_id", "player_1", "player_2"]),
        on="pair_id",
    )
    p1 = player_side(player_path, "1", "player_1")
    p2 = player_side(player_path, "2", "player_2")
    action_context = pl.scan_parquet(action_path)
    utg_seat = (
        action_context.filter(
            (pl.col("street") == "preflop")
            & pl.col("action").is_in(["fold", "call", "raise", "bet", "check", "all_in"])
        )
        .group_by("hand_id")
        .agg(pl.col("player_id").sort_by("action_no").first().alias("utg_player"))
        .join(
            pl.scan_parquet(player_path).select(["hand_id", "player_id", "seat_no"]),
            left_on=["hand_id", "utg_player"],
            right_on=["hand_id", "player_id"],
            how="left",
        )
        .select("hand_id", pl.col("seat_no").cast(pl.Int16).alias("utg_seat"))
    )

    hand = (
        pair_base.join(p1, on=["hand_id", "player_1"])
        .join(p2, on=["hand_id", "player_2"])
        .join(utg_seat, on="hand_id", how="left")
        .with_columns(
            (pl.col("contribution_bb_1") + pl.col("contribution_bb_2")).alias("pair_contribution_bb"),
            (pl.col("contribution_bb_1") - pl.col("contribution_bb_2")).abs().alias("contrib_gap_bb"),
            (pl.col("net_bb_1") - pl.col("net_bb_2")).abs().alias("net_gap_bb"),
            pl.max_horizontal("net_bb_1", "net_bb_2").alias("max_win_bb"),
            (-pl.min_horizontal("net_bb_1", "net_bb_2")).alias("max_loss_bb"),
            pl.min_horizontal((-pl.col("net_bb_1")).clip(lower_bound=0), pl.col("net_bb_2").clip(lower_bound=0)).alias(
                "transfer_1_to_2_bb"
            ),
            pl.min_horizontal((-pl.col("net_bb_2")).clip(lower_bound=0), pl.col("net_bb_1").clip(lower_bound=0)).alias(
                "transfer_2_to_1_bb"
            ),
            (pl.col("went_to_showdown_1") & pl.col("went_to_showdown_2")).cast(pl.Int8).alias("both_showdown"),
            (pl.col("folded_1") ^ pl.col("folded_2")).cast(pl.Int8).alias("one_folded"),
            (pl.col("folded_1") & pl.col("folded_2")).cast(pl.Int8).alias("both_folded"),
            pl.max_horizontal("hole_strength_1", "hole_strength_2").alias("max_hole_strength"),
            (pl.col("hole_strength_1") - pl.col("hole_strength_2")).abs().alias("hole_strength_gap"),
            (pl.col("aggressive_actions_1") + pl.col("aggressive_actions_2")).alias("pair_aggression"),
            (pl.col("raises_1") + pl.col("raises_2")).alias("pair_raises"),
            (pl.col("calls_1") + pl.col("calls_2")).alias("pair_calls"),
            (pl.col("checks_1") + pl.col("checks_2")).alias("pair_checks"),
            (pl.col("postflop_checks_1") + pl.col("postflop_checks_2")).alias("pair_postflop_checks"),
            pl.max_horizontal("max_amount_bb_1", "max_amount_bb_2").alias("max_amount_bb"),
            pl.max_horizontal("max_to_call_bb_1", "max_to_call_bb_2").alias("max_to_call_bb"),
            pl.max_horizontal("max_amount_pot_ratio_1", "max_amount_pot_ratio_2").alias("max_amount_pot_ratio"),
            ((pl.col("seat_no_1") - pl.col("utg_seat") + 18).mod(6)).cast(pl.Int8).alias("pos_1"),
            ((pl.col("seat_no_2") - pl.col("utg_seat") + 18).mod(6)).cast(pl.Int8).alias("pos_2"),
            (pl.col("net_bb_1") < pl.col("net_bb_2")).cast(pl.Int8).alias("p1_is_donor"),
        )
        .with_columns(
            pl.col("pos_1").is_in([2, 3]).cast(pl.Int8).alias("is_late_1"),
            pl.col("pos_2").is_in([2, 3]).cast(pl.Int8).alias("is_late_2"),
            pl.col("pos_1").is_in([4, 5]).cast(pl.Int8).alias("is_blind_1"),
            pl.col("pos_2").is_in([4, 5]).cast(pl.Int8).alias("is_blind_2"),
        )
        .with_columns(
            (
                ((pl.col("is_late_1") == 1) & (pl.col("is_blind_2") == 1))
                | ((pl.col("is_late_2") == 1) & (pl.col("is_blind_1") == 1))
            )
            .cast(pl.Int8)
            .alias("late_vs_blind"),
        )
    )

    members = pl.concat(
        [
            pair_base.select(
                [
                    "pair_id",
                    "hand_id",
                    pl.col("player_1").alias("member"),
                    pl.col("player_2").alias("partner"),
                    pl.lit(1).alias("is_p1"),
                ]
            ),
            pair_base.select(
                [
                    "pair_id",
                    "hand_id",
                    pl.col("player_2").alias("member"),
                    pl.col("player_1").alias("partner"),
                    pl.lit(0).alias("is_p1"),
                ]
            ),
        ]
    )
    fold_at = (
        action_context.filter(pl.col("action") == "fold")
        .group_by(["hand_id", "player_id"])
        .agg(pl.col("action_no").min().alias("partner_fold_no"))
        .rename({"player_id": "partner"})
    )
    member_actions = (
        members.join(action_context, left_on=["hand_id", "member"], right_on=["hand_id", "player_id"], how="left")
        .join(fold_at, on=["hand_id", "partner"], how="left")
        .with_columns(
            (
                (pl.col("players_active") == 2)
                & (pl.col("partner_fold_no").is_null() | (pl.col("partner_fold_no") > pl.col("action_no")))
            ).alias("true_pair_hu")
        )
    )
    interaction = member_actions.group_by(["pair_id", "hand_id"]).agg(
        pl.col("true_pair_hu").sum().alias("true_hu_actions"),
        (pl.col("true_pair_hu") & (pl.col("action") == "check")).sum().alias("true_hu_checks"),
        (pl.col("true_pair_hu") & (pl.col("action") == "call")).sum().alias("true_hu_calls"),
        (pl.col("true_pair_hu") & pl.col("is_aggressive")).sum().alias("true_hu_aggression"),
        (pl.col("true_pair_hu") & (pl.col("street") != "preflop") & (pl.col("action") == "check"))
        .sum()
        .alias("hu_postflop_checks"),
        (pl.col("true_pair_hu") & (pl.col("street").is_in(["turn", "river"])) & (pl.col("action") == "check"))
        .sum()
        .alias("hu_late_checks"),
        (pl.col("true_pair_hu") & (pl.col("action") == "fold") & (pl.col("street") == "flop"))
        .sum()
        .alias("hu_flop_folds"),
        (pl.col("true_pair_hu") & pl.col("action").is_in(["check", "call"]) & (pl.col("is_p1") == 1))
        .sum()
        .alias("p1_hu_passive"),
        (pl.col("true_pair_hu") & pl.col("action").is_in(["check", "call"]) & (pl.col("is_p1") == 0))
        .sum()
        .alias("p2_hu_passive"),
    )
    responses = action_context.filter((pl.col("to_call") > 0) & pl.col("last_aggressor").is_not_null()).select(
        "hand_id",
        pl.col("last_aggressor").alias("member"),
        pl.col("player_id").alias("responder"),
        "action",
        "is_aggressive",
    )
    pressure = (
        members.join(responses, on=["hand_id", "member"], how="inner")
        .with_columns((pl.col("responder") == pl.col("partner")).alias("partner_response"))
        .group_by(["pair_id", "hand_id"])
        .agg(
            (pl.col("partner_response") & (pl.col("action") == "fold")).sum().alias("partner_folds_to_pair"),
            (pl.col("partner_response") & pl.col("action").is_in(["call", "all_in"]) & ~pl.col("is_aggressive"))
            .sum()
            .alias("partner_calls_to_pair"),
            (pl.col("partner_response") & pl.col("is_aggressive")).sum().alias("partner_raises_to_pair"),
            (~pl.col("partner_response") & (pl.col("action") == "fold")).sum().alias("outsider_folds_to_pair"),
            (~pl.col("partner_response") & (pl.col("action") == "call")).sum().alias("outsider_calls_to_pair"),
            (~pl.col("partner_response") & pl.col("is_aggressive")).sum().alias("outsider_raises_to_pair"),
        )
    )

    pf_aggr = action_context.filter((pl.col("street") == "preflop") & pl.col("is_aggressive"))
    opens = pf_aggr.group_by("hand_id").agg(
        pl.col("player_id").sort_by("action_no").first().alias("open_player"),
        pl.col("action_no").sort_by("action_no").first().alias("open_no"),
    )
    threebets = (
        pf_aggr.join(opens, on="hand_id")
        .filter(pl.col("action_no") > pl.col("open_no"))
        .group_by("hand_id")
        .agg(
            pl.col("player_id").sort_by("action_no").first().alias("threebet_player"),
            pl.col("action_no").sort_by("action_no").first().alias("threebet_no"),
        )
    )
    pair_keys = pair_base.select(["pair_id", "hand_id", "player_1", "player_2"])
    pair_open_3bet = (
        pair_keys.join(opens, on="hand_id", how="left")
        .join(threebets, on="hand_id", how="left")
        .with_columns(
            (
                (
                    (pl.col("open_player") == pl.col("player_1"))
                    & (pl.col("threebet_player") == pl.col("player_2"))
                )
                | (
                    (pl.col("open_player") == pl.col("player_2"))
                    & (pl.col("threebet_player") == pl.col("player_1"))
                )
            )
            .fill_null(False)
            .cast(pl.Int8)
            .alias("pair_open_3bet")
        )
        .select(["pair_id", "hand_id", "pair_open_3bet", "threebet_no"])
    )
    out_fold_after_3bet = (
        pair_keys.join(pair_open_3bet.select(["pair_id", "hand_id", "threebet_no"]), on=["pair_id", "hand_id"])
        .join(
            action_context.filter((pl.col("street") == "preflop") & (pl.col("action") == "fold")).select(
                ["hand_id", "player_id", "action_no"]
            ),
            on="hand_id",
        )
        .filter(
            pl.col("threebet_no").is_not_null()
            & (pl.col("action_no") > pl.col("threebet_no"))
            & (pl.col("player_id") != pl.col("player_1"))
            & (pl.col("player_id") != pl.col("player_2"))
        )
        .group_by(["pair_id", "hand_id"])
        .agg(pl.len().cast(pl.Int16).alias("n_out_fold_after_3bet"))
    )

    ctx_names = set(action_context.collect_schema().names())
    spr_aggs = []
    if "spr" in ctx_names:
        spr_aggs += [
            pl.col("spr").min().alias("spr_min"),
            pl.col("spr").median().alias("spr_med"),
        ]
    if "amount_to_bb" in ctx_names:
        spr_aggs += [
            pl.col("amount_to_bb").max().alias("max_amount_to_bb"),
            pl.col("amount_to_bb").filter(pl.col("is_aggressive")).max().alias("max_aggr_amount_to_bb"),
        ]
    pair_spr = None
    if spr_aggs:
        pair_spr = (
            members.join(action_context, left_on=["hand_id", "member"], right_on=["hand_id", "player_id"], how="inner")
            .group_by(["pair_id", "hand_id"])
            .agg(spr_aggs)
        )
    dump_spr = None
    if "spr" in ctx_names:
        dump_spr = (
            members.join(action_context, left_on=["hand_id", "member"], right_on=["hand_id", "player_id"], how="inner")
            .join(hand.select(["pair_id", "hand_id", "p1_is_donor"]), on=["pair_id", "hand_id"])
            .filter(
                ((pl.col("is_p1") == 1) & (pl.col("p1_is_donor") == 1))
                | ((pl.col("is_p1") == 0) & (pl.col("p1_is_donor") == 0))
            )
            .group_by(["pair_id", "hand_id"])
            .agg(pl.col("spr").min().alias("dump_spr"))
        )
    if pair_spr is None:
        pair_spr = pair_base.select(["pair_id", "hand_id"]).with_columns(
            pl.lit(0.0).cast(pl.Float32).alias("spr_min"),
            pl.lit(0.0).cast(pl.Float32).alias("spr_med"),
            pl.lit(0.0).cast(pl.Float32).alias("max_amount_to_bb"),
            pl.lit(0.0).cast(pl.Float32).alias("max_aggr_amount_to_bb"),
        )
    else:
        present = set()
        if "spr" in ctx_names:
            present.update(["spr_min", "spr_med"])
        if "amount_to_bb" in ctx_names:
            present.update(["max_amount_to_bb", "max_aggr_amount_to_bb"])
        pads = [
            pl.lit(0.0).cast(pl.Float32).alias(c)
            for c in ("spr_min", "spr_med", "max_amount_to_bb", "max_aggr_amount_to_bb")
            if c not in present
        ]
        if pads:
            pair_spr = pair_spr.with_columns(pads)
    if dump_spr is None:
        dump_spr = pair_base.select(["pair_id", "hand_id"]).with_columns(
            pl.lit(0.0).cast(pl.Float32).alias("dump_spr")
        )
    board = board_texture(hands)

    fill_cols = [
        "true_hu_actions",
        "true_hu_checks",
        "true_hu_calls",
        "true_hu_aggression",
        "hu_postflop_checks",
        "hu_late_checks",
        "hu_flop_folds",
        "p1_hu_passive",
        "p2_hu_passive",
        "partner_folds_to_pair",
        "partner_calls_to_pair",
        "partner_raises_to_pair",
        "outsider_folds_to_pair",
        "outsider_calls_to_pair",
        "outsider_raises_to_pair",
        "pair_open_3bet",
        "n_out_fold_after_3bet",
        "spr_min",
        "spr_med",
        "max_amount_to_bb",
        "max_aggr_amount_to_bb",
        "dump_spr",
        "board_paired",
        "board_monotone",
        "board_twotone",
        "n_board_cards",
    ]
    result = (
        hand.join(interaction, on=["pair_id", "hand_id"], how="left")
        .join(pressure, on=["pair_id", "hand_id"], how="left")
        .join(pair_open_3bet.select(["pair_id", "hand_id", "pair_open_3bet"]), on=["pair_id", "hand_id"], how="left")
        .join(out_fold_after_3bet, on=["pair_id", "hand_id"], how="left")
        .join(pair_spr, on=["pair_id", "hand_id"], how="left")
        .join(dump_spr, on=["pair_id", "hand_id"], how="left")
        .join(board, on="hand_id", how="left")
        .with_columns(
            [pl.col(c).fill_null(0) for c in fill_cols]
            + [
                pl.col("utg_seat").fill_null(-1).cast(pl.Int16),
                pl.col("pos_1").fill_null(-1).cast(pl.Int8),
                pl.col("pos_2").fill_null(-1).cast(pl.Int8),
                pl.col("late_vs_blind").fill_null(0).cast(pl.Int8),
                pl.col("p1_is_donor").fill_null(0).cast(pl.Int8),
            ]
        )
        .with_columns(
            pl.max_horizontal("transfer_1_to_2_bb", "transfer_2_to_1_bb").alias("transfer_any_bb"),
            (pl.col("transfer_1_to_2_bb") - pl.col("transfer_2_to_1_bb")).abs().alias("flow_oneway"),
            (
                pl.when(pl.col("net_bb_1") <= pl.col("net_bb_2"))
                .then(pl.col("hole_strength_1"))
                .otherwise(pl.col("hole_strength_2"))
                * pl.max_horizontal("transfer_1_to_2_bb", "transfer_2_to_1_bb")
                / 14.0
            )
            .cast(pl.Float32)
            .alias("dump_vs_hole"),
            (
                (pl.col("hole_strength_1") >= 12)
                & (pl.col("hole_strength_2") >= 12)
                & (pl.col("pair_aggression") == 0)
                & (pl.col("both_showdown") == 1)
            )
            .cast(pl.Int8)
            .alias("strong_hole_passive"),
            (pl.col("pair_contribution_bb") / (pl.col("pot_bb_1") + 1e-3)).clip(0, 2).alias("pair_pot_share"),
            (pl.col("transfer_1_to_2_bb") / (pl.col("pot_bb_1") + 1e-3)).clip(0, 1).alias("transfer_pot_ratio"),
            ((pl.col("true_hu_checks") + pl.col("true_hu_calls")) / (pl.col("true_hu_actions") + 1e-3))
            .clip(0, 1)
            .alias("passivity_density"),
            (
                (pl.col("transfer_1_to_2_bb") >= 18.0)
                | ((pl.col("transfer_2_to_1_bb") >= 18.0) & (pl.col("one_folded") == 1))
            )
            .cast(pl.Int8)
            .alias("is_pure_dump"),
            ((pl.col("both_showdown") == 1) & (pl.col("hu_late_checks") >= 1)).cast(pl.Int8).alias("is_soft_checkdown"),
            ((pl.col("outsider_folds_to_pair") >= 1) & (pl.col("pair_raises") >= 1))
            .cast(pl.Int8)
            .alias("is_squeeze_isolation"),
        )
        .with_columns(
            ((pl.col("pair_open_3bet") == 1) & (pl.col("n_out_fold_after_3bet") >= 1))
            .cast(pl.Int8)
            .alias("iso_preflop"),
            ((pl.col("transfer_any_bb") > 3) & (pl.col("partner_calls_to_pair") > 0))
            .cast(pl.Int8)
            .alias("partner_calls_dump"),
            (
                (pl.col("transfer_any_bb") > 0)
                & (
                    ((pl.col("p1_is_donor") == 1) & (pl.col("is_blind_1") == 1))
                    | ((pl.col("p1_is_donor") == 0) & (pl.col("is_blind_2") == 1))
                )
            )
            .cast(pl.Int8)
            .alias("dump_from_oop"),
        )
        .with_columns(
            (
                (
                    pl.col("transfer_any_bb") * 1.5
                    + pl.col("net_gap_bb") * 0.8
                    + 25.0 * pl.col("is_pure_dump")
                    + 15.0 * pl.col("hu_flop_folds")
                    + 12.0 * pl.col("partner_calls_dump")
                ).log1p()
                + 0.3 * pl.col("max_amount_bb").log1p()
            ).alias("directed_signal"),
            (
                (
                    40.0 * pl.col("is_soft_checkdown")
                    + 25.0 * pl.col("both_showdown")
                    + 8.0 * pl.col("hu_late_checks")
                    + 5.0 * pl.col("true_hu_checks")
                ).log1p()
                + 0.3 * pl.col("net_gap_bb").log1p()
            ).alias("soft_signal"),
            (
                (
                    30.0 * pl.col("is_squeeze_isolation")
                    + 22.0 * pl.col("iso_preflop")
                    + 12.0 * pl.col("outsider_folds_to_pair")
                    + 6.0 * pl.col("pair_raises")
                    + 8.0 * pl.col("pair_open_3bet")
                    + pl.col("contrib_gap_bb")
                ).log1p()
            ).alias("isolation_signal"),
            (
                pl.col("transfer_any_bb") * 2.5
                + pl.col("net_gap_bb") * 1.2
                + pl.col("is_pure_dump") * 40.0
                + pl.col("hu_flop_folds") * 25.0
                + pl.col("partner_calls_dump") * 18.0
                + pl.col("pot_bb_1") * 0.3
            ).alias("directed_priority"),
            (
                pl.col("is_soft_checkdown") * 60.0
                + pl.col("both_showdown") * 40.0
                + pl.col("hu_late_checks") * 12.0
                + pl.col("net_gap_bb") * 1.2
                + pl.col("pot_bb_1") * 0.3
            ).alias("soft_priority"),
            (
                pl.col("is_squeeze_isolation") * 45.0
                + pl.col("iso_preflop") * 35.0
                + pl.col("outsider_folds_to_pair") * 15.0
                + pl.col("pair_raises") * 8.0
                + pl.col("contrib_gap_bb") * 2.0
                + pl.col("late_vs_blind") * 8.0
                + pl.col("pot_bb_1") * 0.4
            ).alias("isolation_priority"),
            (pl.col("transfer_any_bb") / (pl.col("transfer_any_bb").max().over("pair_id") + 1e-3)).alias(
                "transfer_to_max"
            ),
            (pl.col("net_gap_bb") / (pl.col("net_gap_bb").max().over("pair_id") + 1e-3)).alias("net_gap_to_max"),
            (pl.col("pot_bb_1") / (pl.col("pot_bb_1").max().over("pair_id") + 1e-3)).alias("pot_to_max"),
            (pl.col("contrib_gap_bb") / (pl.col("contrib_gap_bb").max().over("pair_id") + 1e-3)).alias(
                "contrib_gap_to_max"
            ),
            (pl.col("pair_contribution_bb") / (pl.col("pair_contribution_bb").max().over("pair_id") + 1e-3)).alias(
                "pair_contrib_to_max"
            ),
            (pl.col("outsider_folds_to_pair") / (pl.col("outsider_folds_to_pair").max().over("pair_id") + 1e-3)).alias(
                "outsider_folds_to_max"
            ),
        )
        .select(
            [
                "pair_id",
                "hand_id",
                "table_id",
                "phase_progress",
                "pot_bb_1",
                "pair_contribution_bb",
                "pair_pot_share",
                "contrib_gap_bb",
                "net_gap_bb",
                "max_win_bb",
                "max_loss_bb",
                "transfer_1_to_2_bb",
                "transfer_2_to_1_bb",
                "transfer_any_bb",
                "flow_oneway",
                "dump_vs_hole",
                "strong_hole_passive",
                "transfer_pot_ratio",
                "passivity_density",
                "hole_strength_gap",
                "both_showdown",
                "one_folded",
                "both_folded",
                "max_hole_strength",
                "pair_aggression",
                "pair_raises",
                "pair_calls",
                "pair_checks",
                "pair_postflop_checks",
                "max_amount_bb",
                "max_to_call_bb",
                "max_amount_pot_ratio",
                "is_pure_dump",
                "is_soft_checkdown",
                "is_squeeze_isolation",
                "iso_preflop",
                "pair_open_3bet",
                "partner_calls_dump",
                "late_vs_blind",
                "dump_from_oop",
                "n_out_fold_after_3bet",
                *[c for c in fill_cols if c not in {"pair_open_3bet", "n_out_fold_after_3bet"}],
                "directed_signal",
                "soft_signal",
                "isolation_signal",
                "directed_priority",
                "soft_priority",
                "isolation_priority",
                "transfer_to_max",
                "net_gap_to_max",
                "pot_to_max",
                "contrib_gap_to_max",
                "pair_contrib_to_max",
                "outsider_folds_to_max",
            ]
        )
        .rename({"pot_bb_1": "pot_bb"})
    )
    result.sink_parquet(out_path)


### 8.4 Partner-field and pair aggregate

`EVIDENCE_ONLY_COLS` do not go through `mean/max/p95/top5`. Pair X adds `isolation_purity_rate`, `transfer_roll20_max`, `isolation_roll20_max`, then table percentile.


In [ ]:
def _partner_field_side(
    pair_keys: pl.LazyFrame,
    occ: pl.LazyFrame,
    stats: pl.LazyFrame,
    prefix: str,
    player_col: str,
    partner_col: str,
    cols: list[str],
) -> pl.DataFrame:
    seated = (
        pair_keys.join(occ, left_on=[player_col, "table_id"], right_on=["player_id", "table_id"])
        .join(
            occ.select(["hand_id", "player_id", pl.lit(1).alias("_present")]),
            left_on=["hand_id", partner_col],
            right_on=["hand_id", "player_id"],
            how="left",
        )
        .with_columns(pl.col("_present").is_not_null().alias("_with"))
        .join(stats, left_on=["hand_id", player_col], right_on=["hand_id", "player_id"], how="left")
        .group_by(["pair_id", "_with"])
        .agg(
            pl.len().alias(f"{prefix}_n"),
            *[pl.col(c).cast(pl.Float64).mean().alias(f"{prefix}_{c}") for c in cols],
        )
        .collect(engine="streaming")
    )
    with_p = seated.filter(pl.col("_with")).drop("_with").rename(
        {c: f"{c}_with" for c in seated.columns if c not in ("pair_id", "_with")}
    )
    field = seated.filter(~pl.col("_with")).drop("_with").rename(
        {c: f"{c}_field" for c in seated.columns if c not in ("pair_id", "_with")}
    )
    keys = seated.select("pair_id").unique()
    return keys.join(with_p, on="pair_id", how="left").join(field, on="pair_id", how="left")


def add_partner_field_contrasts(
    df: pl.DataFrame,
    seats: pl.LazyFrame,
    hands: pl.LazyFrame,
    player_path: Path,
    phase: str,
) -> pl.DataFrame:
    if not {"pair_id", "player_1", "player_2", "table_id"}.issubset(df.columns):
        return df
    occ = (
        seats.join(hands.select(["hand_id", "table_id", "phase"]), on="hand_id")
        .filter(pl.col("phase") == phase)
        .select(["hand_id", "table_id", "player_id"])
    )
    available = set(pl.scan_parquet(player_path).collect_schema().names())
    cols = [c for c in FIELD_COLS if c in available]
    if not cols:
        return df
    stats = pl.scan_parquet(player_path).select(["hand_id", "player_id", *cols])
    pair_keys = df.select(["pair_id", "player_1", "player_2", "table_id"]).unique().lazy()
    p1 = _partner_field_side(pair_keys, occ, stats, "p1", "player_1", "player_2", cols)
    p2 = _partner_field_side(pair_keys, occ, stats, "p2", "player_2", "player_1", cols)
    out = df.join(p1, on="pair_id", how="left").join(p2, on="pair_id", how="left")
    exprs = []
    for prefix in ("p1", "p2"):
        n_with = f"{prefix}_n_with"
        n_field = f"{prefix}_n_field"
        if n_with in out.columns and n_field in out.columns:
            exprs.append(
                (pl.col(n_with).fill_null(0) / (pl.col(n_with).fill_null(0) + pl.col(n_field).fill_null(0) + 1.0))
                .cast(pl.Float32)
                .alias(f"{prefix}_with_share")
            )
        for col in cols:
            w, f = f"{prefix}_{col}_with", f"{prefix}_{col}_field"
            if w not in out.columns or f not in out.columns:
                continue
            exprs += [
                (pl.col(w) - pl.col(f)).cast(pl.Float32).alias(f"{prefix}_{col}_contrast"),
                ((pl.col(w) - pl.col(f)) * (pl.col(n_with).fill_null(0) / (pl.col(n_with).fill_null(0) + 25.0)))
                .cast(pl.Float32)
                .alias(f"{prefix}_{col}_contrast_shrunk"),
                ((pl.col(w) - pl.col(f)) * (pl.col(n_with).fill_null(0) + 1.0).sqrt())
                .cast(pl.Float32)
                .alias(f"{prefix}_{col}_contrast_t"),
                ((pl.col(w) + 0.05) / (pl.col(f) + 0.05)).cast(pl.Float32).alias(f"{prefix}_{col}_ratio"),
            ]
    out = out.with_columns(exprs)
    extra = []
    if "p1_net_bb_contrast" in out.columns and "p2_net_bb_contrast" in out.columns:
        extra += [
            (pl.col("p1_net_bb_contrast") + pl.col("p2_net_bb_contrast")).alias("pair_net_contrast"),
            (pl.col("p1_net_bb_contrast") - pl.col("p2_net_bb_contrast")).abs().alias("directed_asymmetry"),
        ]
    if "p1_aggressive_actions_contrast" in out.columns and "p2_aggressive_actions_contrast" in out.columns:
        extra += [
            (-(pl.col("p1_aggressive_actions_contrast") + pl.col("p2_aggressive_actions_contrast"))).alias(
                "soft_contrast"
            ),
            (pl.col("p1_aggressive_actions_contrast") + pl.col("p2_aggressive_actions_contrast")).alias(
                "pair_agr_contrast"
            ),
        ]
    if "p1_raises_contrast" in out.columns and "p2_raises_contrast" in out.columns:
        extra.append((pl.col("p1_raises_contrast") + pl.col("p2_raises_contrast")).alias("pair_raise_contrast"))
    if "p1_net_bb_contrast_shrunk" in out.columns and "p2_net_bb_contrast_shrunk" in out.columns:
        extra += [
            (pl.col("p1_net_bb_contrast_shrunk") + pl.col("p2_net_bb_contrast_shrunk")).alias(
                "pair_net_contrast_shrunk"
            ),
            (pl.col("p1_net_bb_contrast_shrunk") - pl.col("p2_net_bb_contrast_shrunk")).abs().alias(
                "directed_asymmetry_shrunk"
            ),
        ]
    if "p1_aggressive_actions_ratio" in out.columns and "p2_aggressive_actions_ratio" in out.columns:
        extra.append(
            (pl.col("p1_aggressive_actions_ratio") * pl.col("p2_aggressive_actions_ratio")).alias("pair_agr_ratio_prod")
        )
    if "p1_net_bb_ratio" in out.columns and "p2_net_bb_ratio" in out.columns:
        extra.append((pl.col("p1_net_bb_ratio") / (pl.col("p2_net_bb_ratio") + 0.01)).alias("pair_net_ratio_quot"))
    if extra:
        out = out.with_columns(extra)
    drop_raw = [
        c
        for c in out.columns
        if (c.endswith("_with") or c.endswith("_field")) and not c.endswith("_n_with") and not c.endswith("_n_field")
    ]
    return out.drop(drop_raw) if drop_raw else out


def aggregate_pair_features(
    hand_path: Path,
    pairs: pl.DataFrame,
    players: pl.LazyFrame,
    baseline_path: Path,
    hand_features: list[str],
    score_features: list[str],
    tail_features: list[str],
    score_q95: dict[str, float],
) -> pl.DataFrame:
    lf = pl.scan_parquet(hand_path)
    agg = [pl.len().alias("shared_hands_calc")]
    agg += [pl.col(c).mean().alias(f"{c}_mean") for c in hand_features]
    agg += [pl.col(c).max().alias(f"{c}_max") for c in hand_features]
    agg += [pl.col(c).quantile(0.95, interpolation="nearest").alias(f"{c}_p95") for c in hand_features]
    agg += [pl.col(c).top_k(5).mean().alias(f"{c}_top5") for c in tail_features]
    agg += [(pl.col(c) >= score_q95[c]).mean().alias(f"{c}_rate95") for c in score_features if c in score_q95]
    agg += [
        pl.col("transfer_1_to_2_bb").sum().alias("_cum_t12"),
        pl.col("transfer_2_to_1_bb").sum().alias("_cum_t21"),
        pl.col("net_gap_bb").sum().alias("_cum_net_gap"),
        pl.col("pot_bb").sum().alias("_cum_pot"),
        pl.col("true_hu_checks").sum().alias("_cum_hu_checks"),
        pl.col("true_hu_actions").sum().alias("_cum_hu_actions"),
        pl.col("both_showdown").sum().alias("_cum_showdown"),
        pl.col("outsider_folds_to_pair").sum().alias("_cum_outsider_folds"),
        pl.col("outsider_calls_to_pair").sum().alias("_cum_outsider_calls"),
        pl.col("partner_folds_to_pair").sum().alias("_cum_partner_folds"),
        pl.col("partner_calls_to_pair").sum().alias("_cum_partner_calls"),
        pl.col("partner_raises_to_pair").sum().alias("_cum_partner_raises"),
        pl.col("pair_raises").sum().alias("_cum_raises"),
        pl.col("pair_contribution_bb").sum().alias("_cum_pair_contrib"),
        pl.col("contrib_gap_bb").sum().alias("_cum_contrib_gap"),
        pl.col("hu_flop_folds").sum().alias("_cum_flop_folds"),
        pl.col("hu_postflop_checks").sum().alias("_cum_hu_postflop_checks"),
        pl.col("hu_late_checks").sum().alias("_cum_late_checks"),
        pl.col("is_pure_dump").sum().alias("_cum_pure_dump"),
        pl.col("is_soft_checkdown").sum().alias("_cum_soft_checkdown"),
        pl.col("is_squeeze_isolation").sum().alias("_cum_squeeze_isolation"),
        pl.col("table_id").first().alias("table_id"),
        (
            ((pl.col("outsider_folds_to_pair") >= 1) & (pl.col("partner_folds_to_pair") == 0))
            .cast(pl.Float32)
            .mean()
            .alias("isolation_purity_rate")
        ),
    ]
    pair_stats = (
        lf.group_by("pair_id")
        .agg(agg)
        .with_columns(
            ((pl.col("_cum_t12") - pl.col("_cum_t21")).abs() / (pl.col("_cum_t12") + pl.col("_cum_t21") + 1e-3)).alias(
                "transfer_imbalance"
            ),
            ((pl.col("_cum_t12") + pl.col("_cum_t21")) / (pl.col("shared_hands_calc") + 1e-3)).alias("transfer_per_hand"),
            (pl.col("_cum_net_gap") / (pl.col("_cum_pot") + 1e-3)).alias("net_flow_ratio"),
            (pl.col("_cum_hu_checks") / (pl.col("_cum_hu_actions") + 1e-3)).alias("showdown_check_ratio"),
            (pl.col("_cum_showdown") / (pl.col("shared_hands_calc") + 1e-3)).alias("both_showdown_rate"),
            (pl.col("_cum_outsider_folds") / (pl.col("shared_hands_calc") + 1e-3)).alias("outsider_fold_rate"),
            (
                (pl.col("_cum_outsider_folds") - pl.col("_cum_partner_folds"))
                / (pl.col("_cum_outsider_folds") + pl.col("_cum_partner_folds") + 1e-3)
            ).alias("isolation_imbalance"),
            (pl.col("_cum_contrib_gap") / (pl.col("_cum_pair_contrib") + 1e-3)).alias("contrib_asymmetry_ratio"),
            (pl.col("_cum_flop_folds") / (pl.col("shared_hands_calc") + 1e-3)).alias("donor_flop_fold_rate"),
            (pl.col("_cum_late_checks") / (pl.col("shared_hands_calc") + 1e-3)).alias("late_checkdown_rate"),
            (pl.col("_cum_pure_dump") / (pl.col("shared_hands_calc") + 1e-3)).alias("pure_dump_rate"),
            (pl.col("_cum_soft_checkdown") / (pl.col("shared_hands_calc") + 1e-3)).alias("soft_checkdown_rate"),
            (pl.col("_cum_squeeze_isolation") / (pl.col("shared_hands_calc") + 1e-3)).alias("squeeze_isolation_rate"),
            (pl.col("_cum_outsider_folds") / (pl.col("_cum_outsider_folds") + pl.col("_cum_outsider_calls") + 1e-3)).alias(
                "outsider_fold_efficiency"
            ),
            (
                pl.col("_cum_partner_folds")
                / (pl.col("_cum_partner_folds") + pl.col("_cum_partner_calls") + pl.col("_cum_partner_raises") + 1e-3)
            ).alias("partner_immunity_ratio"),
            (pl.col("_cum_hu_postflop_checks") / (pl.col("_cum_hu_actions") + 1e-3)).alias("hu_postflop_check_ratio"),
            (pl.col("transfer_any_bb_max") / (pl.col("shared_hands_calc").log1p() + 1e-3)).alias("transfer_max_log_norm"),
            (pl.col("net_gap_bb_max") / (pl.col("shared_hands_calc").log1p() + 1e-3)).alias("net_gap_max_log_norm"),
            (pl.col("pot_bb_max") / (pl.col("shared_hands_calc").log1p() + 1e-3)).alias("pot_max_log_norm"),
            (pl.col("contrib_gap_bb_max") / (pl.col("shared_hands_calc").log1p() + 1e-3)).alias("contrib_gap_max_log_norm"),
            (pl.col("pair_contribution_bb_max") / (pl.col("shared_hands_calc").log1p() + 1e-3)).alias(
                "pair_contrib_max_log_norm"
            ),
        )
        .drop(
            [
                "_cum_t12",
                "_cum_t21",
                "_cum_net_gap",
                "_cum_pot",
                "_cum_hu_checks",
                "_cum_hu_actions",
                "_cum_showdown",
                "_cum_outsider_folds",
                "_cum_outsider_calls",
                "_cum_partner_folds",
                "_cum_partner_calls",
                "_cum_partner_raises",
                "_cum_raises",
                "_cum_pair_contrib",
                "_cum_contrib_gap",
                "_cum_flop_folds",
                "_cum_hu_postflop_checks",
                "_cum_late_checks",
                "_cum_pure_dump",
                "_cum_soft_checkdown",
                "_cum_squeeze_isolation",
            ]
        )
    )
    burst = (
        lf.with_columns(((pl.col("phase_progress") * 8).floor().clip(0, 7)).cast(pl.Int8).alias("_phase_bin"))
        .group_by(["pair_id", "_phase_bin"])
        .agg(
            pl.col("directed_signal").mean().alias("_d_burst"),
            pl.col("soft_signal").mean().alias("_s_burst"),
            pl.col("isolation_signal").mean().alias("_i_burst"),
        )
        .group_by("pair_id")
        .agg(
            pl.col("_d_burst").max().alias("directed_burst_max"),
            pl.col("_s_burst").max().alias("soft_burst_max"),
            pl.col("_i_burst").max().alias("isolation_burst_max"),
            (pl.col("_d_burst").max() - pl.col("_d_burst").mean()).alias("directed_burst_excess"),
            (pl.col("_s_burst").max() - pl.col("_s_burst").mean()).alias("soft_burst_excess"),
            (pl.col("_i_burst").max() - pl.col("_i_burst").mean()).alias("isolation_burst_excess"),
        )
    )
    conc = (
        lf.group_by("pair_id")
        .agg(
            pl.col("transfer_any_bb").sum().alias("_t_sum"),
            pl.col("transfer_any_bb").top_k(3).sum().alias("_t_top3"),
            pl.col("transfer_any_bb").top_k(5).sum().alias("_t_top5"),
            pl.col("net_gap_bb").sum().alias("_ng_sum"),
            pl.col("net_gap_bb").top_k(5).sum().alias("_ng_top5"),
        )
        .with_columns(
            (pl.col("_t_top3") / (pl.col("_t_sum") + 1e-3)).alias("transfer_top3_share"),
            (pl.col("_t_top5") / (pl.col("_t_sum") + 1e-3)).alias("transfer_top5_share"),
            (pl.col("_ng_top5") / (pl.col("_ng_sum") + 1e-3)).alias("net_gap_top5_share"),
        )
        .drop(["_t_sum", "_t_top3", "_t_top5", "_ng_sum", "_ng_top5"])
    )
    roll20 = (
        lf.sort(["pair_id", "phase_progress"])
        .with_columns(
            rolling_sum_pair("transfer_any_bb", "_t_roll20"),
            rolling_sum_pair("isolation_signal", "_i_roll20"),
        )
        .group_by("pair_id")
        .agg(
            pl.col("_t_roll20").max().alias("transfer_roll20_max"),
            pl.col("_i_roll20").max().alias("isolation_roll20_max"),
        )
    )
    meta = players.select([c for c in META_COLS if c in players.collect_schema().names()])
    have_meta = set(meta.collect_schema().names())
    m1 = meta.rename({c: ("player_1" if c == "player_id" else f"{c}_1") for c in have_meta})
    m2 = meta.rename({c: ("player_2" if c == "player_id" else f"{c}_2") for c in have_meta})
    player_base = (
        pl.scan_parquet(baseline_path)
        .select(BASE_PLAYER_COLS)
        .group_by("player_id")
        .agg(
            pl.col("hands").sum().alias("total_hands"),
            pl.col("base_aggression").mean().alias("base_aggression"),
            pl.col("base_showdown").mean().alias("base_showdown"),
            pl.col("base_calls").mean().alias("base_calls"),
            pl.col("base_checks").mean().alias("base_checks"),
        )
    )
    pb_keep = ["player_id", "total_hands", "base_aggression", "base_showdown", "base_calls", "base_checks"]
    pb1 = player_base.rename({c: ("player_1" if c == "player_id" else f"{c}_1") for c in pb_keep})
    pb2 = player_base.rename({c: ("player_2" if c == "player_id" else f"{c}_2") for c in pb_keep})
    pair_meta = pairs.lazy().select(["pair_id", "player_1", "player_2"]).join(pb1, on="player_1", how="left").join(
        pb2, on="player_2", how="left"
    )
    if "account_age_days" in have_meta:
        pair_meta = pair_meta.join(m1, on="player_1", how="left").join(m2, on="player_2", how="left")
        pair_meta = pair_meta.select(
            "pair_id",
            (pl.col("account_age_days_1").cast(pl.Float32) - pl.col("account_age_days_2").cast(pl.Float32))
            .abs()
            .alias("account_age_gap"),
            (pl.col("experience_hands_bucket_1") == pl.col("experience_hands_bucket_2")).cast(pl.Int8).alias(
                "same_experience"
            ),
            (pl.col("preferred_stake_1") == pl.col("preferred_stake_2")).cast(pl.Int8).alias("same_stake"),
            (pl.col("region_bucket_1") == pl.col("region_bucket_2")).cast(pl.Int8).alias("same_region"),
            (pl.col("client_family_1") == pl.col("client_family_2")).cast(pl.Int8).alias("same_client"),
            pl.col("total_hands_1").fill_null(0).alias("player_hands_1"),
            pl.col("total_hands_2").fill_null(0).alias("player_hands_2"),
            pl.col("base_aggression_1").fill_null(0.0).alias("base_aggression_1"),
            pl.col("base_aggression_2").fill_null(0.0).alias("base_aggression_2"),
            pl.col("base_showdown_1").fill_null(0.0).alias("base_showdown_1"),
            pl.col("base_showdown_2").fill_null(0.0).alias("base_showdown_2"),
        )
    else:
        pair_meta = pair_meta.select(
            "pair_id",
            pl.col("total_hands_1").fill_null(0).alias("player_hands_1"),
            pl.col("total_hands_2").fill_null(0).alias("player_hands_2"),
            pl.col("base_aggression_1").fill_null(0.0).alias("base_aggression_1"),
            pl.col("base_aggression_2").fill_null(0.0).alias("base_aggression_2"),
            pl.col("base_showdown_1").fill_null(0.0).alias("base_showdown_1"),
            pl.col("base_showdown_2").fill_null(0.0).alias("base_showdown_2"),
        )

    base_cols = [c for c in ["pair_id", "player_1", "player_2", "shared_hands"] if c in pairs.columns]
    rank_metrics = [c for c in TABLE_RANK_METRICS]
    out = (
        pairs.lazy()
        .select(base_cols)
        .join(pair_stats, on="pair_id", how="left")
        .join(burst, on="pair_id", how="left")
        .join(conc, on="pair_id", how="left")
        .join(roll20, on="pair_id", how="left")
        .join(pair_meta, on="pair_id", how="left")
        .with_columns(
            pl.col("isolation_purity_rate").fill_null(0.0),
            pl.col("transfer_roll20_max").fill_null(0.0),
            pl.col("isolation_roll20_max").fill_null(0.0),
            (pl.col("shared_hands_calc") / (pl.col("player_hands_1") + 1e-3)).clip(0, 1).alias("overlap_share_1"),
            (pl.col("shared_hands_calc") / (pl.col("player_hands_2") + 1e-3)).clip(0, 1).alias("overlap_share_2"),
            (pl.col("both_showdown_rate") / (pl.col("base_showdown_1") * pl.col("base_showdown_2") + 1e-4)).alias(
                "showdown_excess_ratio"
            ),
            (
                pl.col("transfer_per_hand")
                / ((pl.col("base_aggression_1") + pl.col("base_aggression_2")) / 2.0 + 1e-3)
            ).alias("transfer_vs_aggression"),
            (pl.col("shared_hands_calc") / (pl.col("player_hands_1") * pl.col("player_hands_2") + 1e-3).sqrt())
            .clip(0, 1)
            .alias("co_presence_ratio"),
            (
                pl.col("shared_hands_calc") / ((pl.col("player_hands_1") + pl.col("player_hands_2")) / 2.0 + 1e-3)
            )
            .clip(0, 1)
            .alias("geometric_overlap"),
            (pl.col("shared_hands_calc") / pl.min_horizontal("player_hands_1", "player_hands_2").clip(1, None))
            .clip(0, 1)
            .alias("co_sitting_affinity"),
        )
        .with_columns(
            pl.max_horizontal("overlap_share_1", "overlap_share_2").alias("max_overlap_share"),
            pl.min_horizontal("overlap_share_1", "overlap_share_2").alias("min_overlap_share"),
        )
        .collect(engine="streaming")
    )
    present_rank = [c for c in rank_metrics if c in out.columns]
    extra_rank = [c for c in ("max_overlap_share", "geometric_overlap", "co_sitting_affinity") if c in out.columns]
    if "table_id" in out.columns and present_rank:
        out = out.with_columns(
            [
                (pl.col(c).rank(method="average").over("table_id") / (pl.len().over("table_id") + 1e-3))
                .cast(pl.Float32)
                .alias(f"{c}_table_pct")
                for c in present_rank + extra_rank
            ]
        )
    return out.drop(
        [c for c in ("base_aggression_1", "base_aggression_2", "base_showdown_1", "base_showdown_2") if c in out.columns]
    )


### 8.5 Evidence engines

MAP@5, family quad (HGB+LGB+XGB+XGBRanker), `predict_quad`. Z-blend helpers remain in the file but are **not called** at inference.


In [ ]:
def matrix_from(df: pl.DataFrame, cols: list[str]) -> pd.DataFrame:
    return df.select(cols).to_pandas().replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)


def top_mask(scores: np.ndarray, rate: float) -> np.ndarray:
    n = max(1, int(round(len(scores) * rate)))
    mask = np.zeros(len(scores), dtype=bool)
    mask[np.argsort(-scores)[:n]] = True
    return mask


def blend_within_pair(part: pl.DataFrame, model: np.ndarray, heur: np.ndarray, w_model: float = 0.85) -> np.ndarray:
    scored = part.select("pair_id").with_columns(
        pl.Series("_m", model.astype(np.float32)),
        pl.Series("_h", heur.astype(np.float32)),
    )
    scored = scored.with_columns(
        ((pl.col("_m") - pl.col("_m").mean().over("pair_id")) / (pl.col("_m").std().over("pair_id") + 1e-6)).alias(
            "_mz"
        ),
        ((pl.col("_h") - pl.col("_h").mean().over("pair_id")) / (pl.col("_h").std().over("pair_id") + 1e-6)).alias(
            "_hz"
        ),
        pl.col("_m").std().over("pair_id").fill_null(0).alias("_mstd"),
    )
    return np.where(
        scored["_mstd"].to_numpy() < 1e-6,
        model,
        w_model * scored["_mz"].to_numpy() + (1.0 - w_model) * scored["_hz"].to_numpy(),
    ).astype(np.float32)


def official_evidence_map5(pair_ids, hand_ids, scores, truth_dict) -> float:
    frame = pd.DataFrame({"pair_id": pair_ids, "hand_id": hand_ids, "score": scores})
    res = []
    for p, grp in frame.groupby("pair_id", sort=False):
        relevant = truth_dict.get(p, set())
        if not relevant:
            continue
        preds = grp.sort_values("score", ascending=False, kind="mergesort")["hand_id"].to_list()[:5]
        hits = 0
        psum = 0.0
        for r, h in enumerate(preds, start=1):
            if h in relevant:
                hits += 1
                psum += hits / r
        res.append(psum / min(len(relevant), 5))
    return float(np.mean(res)) if res else 0.0


def family_priority(part: pl.DataFrame, bid: np.ndarray) -> np.ndarray:
    d = part["directed_priority"].to_numpy() if "directed_priority" in part.columns else part["directed_signal"].to_numpy()
    s = part["soft_priority"].to_numpy() if "soft_priority" in part.columns else part["soft_signal"].to_numpy()
    i = (
        part["isolation_priority"].to_numpy()
        if "isolation_priority" in part.columns
        else part["isolation_signal"].to_numpy()
    )
    return np.where(bid == 1, d, np.where(bid == 2, s, i)).astype(np.float32)


def fit_family_quad(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray | None, seed: int):
    hgb = HistGradientBoostingClassifier(
        max_iter=180,
        learning_rate=0.045,
        max_leaf_nodes=30,
        min_samples_leaf=15,
        l2_regularization=2.5,
        random_state=seed,
    )
    hgb.fit(X, y)
    lgb = LGBMClassifier(
        n_estimators=180,
        learning_rate=0.045,
        num_leaves=31,
        min_child_samples=15,
        reg_lambda=2.5,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )
    lgb.fit(X, y)
    xgb = XGBClassifier(
        n_estimators=180,
        learning_rate=0.045,
        max_depth=5,
        min_child_weight=2,
        reg_lambda=2.5,
        random_state=seed,
        tree_method="hist",
        n_jobs=-1,
    )
    xgb.fit(X, y)
    ranker = XGBRanker(
        n_estimators=200,
        learning_rate=0.035,
        max_depth=4,
        objective="rank:pairwise",
        eval_metric="map@5",
        tree_method="hist",
        random_state=seed,
        n_jobs=-1,
    )
    if groups is not None:
        ranker.fit(X, y, group=groups)
    else:
        ranker.fit(X, y, group=np.array([len(y)], dtype=np.int32))
    return hgb, lgb, xgb, ranker


def predict_quad(models, X: pd.DataFrame) -> np.ndarray:
    hgb, lgb, xgb, ranker = models
    trees = (hgb.predict_proba(X)[:, 1] + lgb.predict_proba(X)[:, 1] + xgb.predict_proba(X)[:, 1]) / 3.0
    return (0.80 * trees + 0.20 * ranker.predict(X)).astype(np.float32)


### 8.6 `main()` — train, OOF, infer

Pair ensemble `0.40 XGB + 0.35 LGB + 0.25 Cat`. Isolation detector weight 10. Evidence `0.60 spec + 0.40 global` on **every** eval pair, chunks of 8,000.


In [ ]:
def main() -> None:
    if PREP_DIR.exists():
        shutil.rmtree(PREP_DIR)
    PREP_DIR.mkdir(parents=True, exist_ok=True)
    data_dir = find_data_dir()
    print(f"Competition data: {data_dir}")
    print(f"Scratch features (rebuilt every run): {PREP_DIR}")

    paths = {
        "dev_pairs": PREP_DIR / "dev_pairs.parquet",
        "eval_pairs": PREP_DIR / "eval_pairs.parquet",
        "dev_pair_hands": PREP_DIR / "dev_pair_hands.parquet",
        "eval_pair_hands": PREP_DIR / "eval_pair_hands.parquet",
        "action": PREP_DIR / "action_context.parquet",
        "player": PREP_DIR / "player_hand_features.parquet",
        "baseline": PREP_DIR / "player_baselines.parquet",
        "dev_hand": PREP_DIR / "dev_hand_features.parquet",
        "eval_hand": PREP_DIR / "eval_hand_features.parquet",
    }

    hands = pl.scan_parquet(data_dir / "hands.parquet")
    seats = pl.scan_parquet(data_dir / "seats.parquet")
    actions = pl.scan_parquet(data_dir / "actions.parquet")
    players = pl.scan_parquet(data_dir / "players.parquet")
    labels = pl.read_csv(data_dir / "development_labels.csv")
    evidence = pl.read_csv(data_dir / "development_evidence.csv")
    eval_pairs = pl.read_csv(data_dir / "evaluation_pairs.csv")
    sample_sub = pl.read_csv(data_dir / "sample_submission.csv")
    print(labels.group_by(["label", "behavior_family"]).len().sort(["label", "behavior_family"]))

    dev_pairs, eval_pairs_prep = build_pair_indexes(seats, hands, labels, eval_pairs, paths)
    print(f"Development pairs: {len(dev_pairs):,} ({int(dev_pairs['is_pu'].sum()):,} PU)")
    print(f"Evaluation pairs:  {len(eval_pairs_prep):,}")

    build_action_context(actions, hands, paths["action"])
    act_names = list(pl.scan_parquet(paths["action"]).collect_schema().names())
    print(f"Action context extras: {[c for c in ('spr', 'amount_to_bb') if c in act_names] or '(none)'}")
    print(f"Board columns: {board_card_columns(list(hands.collect_schema().names())) or '(none)'}")
    build_player_hands(seats, hands, paths["action"], paths["player"], paths["baseline"])
    for pair_hands, pairs_path, out, name in (
        (paths["dev_pair_hands"], paths["dev_pairs"], paths["dev_hand"], "Development"),
        (paths["eval_pair_hands"], paths["eval_pairs"], paths["eval_hand"], "Evaluation"),
    ):
        build_pair_hand_features(pair_hands, pairs_path, paths["player"], paths["action"], out, hands)
        n = pl.scan_parquet(out).select(pl.len()).collect().item()
        print(f"{name} pair-hands: {n:,}")

    skip = {"pair_id", "hand_id", "player_1", "player_2", "p_low", "p_high", "table_id"}
    schema = pl.scan_parquet(paths["dev_hand"]).collect_schema()
    hand_features = [
        c
        for c, dtype in schema.items()
        if dtype.is_numeric() and c not in skip and "label" not in c and "behavior_id" not in c
    ]
    pair_agg_features = [c for c in hand_features if c not in EVIDENCE_ONLY_COLS]
    score_features = [c for c in pair_agg_features if c.endswith("_signal") or c.endswith("_priority")]
    tail_tokens = (
        "signal",
        "priority",
        "score",
        "pot_bb",
        "contribution",
        "net_gap",
        "amount",
        "raise",
        "partner",
        "to_max",
        "iso_preflop",
        "dump",
        "late",
        "3bet",
    )
    tail_features = list(
        dict.fromkeys(score_features + [c for c in pair_agg_features if any(t in c for t in tail_tokens)])
    )[:48]
    score_q95 = (
        pl.scan_parquet(paths["dev_hand"]).select([pl.col(c).quantile(0.95).alias(c) for c in score_features]).collect()
        .row(0, named=True)
        if score_features
        else {}
    )

    print("Aggregating pair features...")
    print(f"Pair-X hand cols: {len(pair_agg_features)} | evidence-only: {sorted(EVIDENCE_ONLY_COLS & set(hand_features))}")
    dev_features = aggregate_pair_features(
        paths["dev_hand"],
        dev_pairs,
        players,
        paths["baseline"],
        pair_agg_features,
        score_features,
        tail_features,
        score_q95,
    )
    dev_features = add_partner_field_contrasts(dev_features, seats, hands, paths["player"], "development")

    pair_groups = (
        pl.scan_parquet(paths["dev_hand"])
        .select(["pair_id", "hand_id"])
        .join(hands.select(["hand_id", "table_id"]), on="hand_id")
        .group_by(["pair_id", "table_id"])
        .agg(pl.len().alias("_n"))
        .sort(["pair_id", "_n"], descending=[False, True])
        .unique("pair_id", keep="first", maintain_order=True)
        .select("pair_id", pl.col("table_id").alias("cv_group"))
        .collect()
    )

    known_info = labels.select(["pair_id", "behavior_family"]).with_columns(pl.lit(True).alias("is_known"))
    train_df = dev_features.join(known_info, on="pair_id", how="left").join(pair_groups, on="pair_id", how="left").sort(
        "pair_id"
    )
    known = train_df["is_known"].fill_null(False).to_numpy().astype(bool)
    behavior_y = np.array(
        [BEHAVIOR_TO_ID.get(x, 0) for x in train_df["behavior_family"].fill_null("none").to_list()],
        dtype=np.int8,
    )
    y = (behavior_y > 0).astype(np.int8)
    exclude = {"pair_id", "player_1", "player_2", "cv_group", "is_known", "table_id"}
    feature_cols = [c for c, dtype in train_df.schema.items() if dtype.is_numeric() and c not in exclude]
    X = matrix_from(train_df, feature_cols)
    groups = train_df["cv_group"].fill_null("NA").to_numpy()
    pu_expansion = max(1.0, len(eval_pairs_prep) / max((~known).sum(), 1))
    stress_weight = np.where(known, 1.0, pu_expansion)
    fit_weight = np.where(y == 1, 2.0, np.where(known, 1.0, 0.35)).astype(np.float32)
    print(f"Train matrix: {X.shape[0]:,} × {X.shape[1]} | pos={int(y.sum()):,} PU-like={int((~known).sum()):,}")

    xgb_params = dict(
        n_estimators=1600,
        learning_rate=0.022,
        max_depth=6,
        min_child_weight=6,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.2,
        reg_lambda=8.0,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        early_stopping_rounds=120,
        n_jobs=-1,
    )
    lgb_params = dict(
        n_estimators=1200,
        learning_rate=0.025,
        num_leaves=45,
        min_child_samples=18,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.2,
        reg_lambda=8.0,
        objective="binary",
        n_jobs=-1,
        verbose=-1,
    )
    cb_params = dict(
        iterations=1100,
        learning_rate=0.030,
        depth=6,
        l2_leaf_reg=6.0,
        loss_function="Logloss",
        eval_metric="Logloss",
        verbose=0,
        thread_count=-1,
        allow_writing_files=False,
    )
    behavior_params = dict(
        n_estimators=800,
        learning_rate=0.025,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_alpha=0.15,
        reg_lambda=6.0,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        early_stopping_rounds=80,
        n_jobs=-1,
    )

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    n = len(train_df)
    oof_xgb = np.zeros(n, dtype=np.float32)
    oof_lgb = np.zeros(n, dtype=np.float32)
    oof_cb = np.zeros(n, dtype=np.float32)
    oof_dir = np.zeros(n, dtype=np.float32)
    oof_soft = np.zeros(n, dtype=np.float32)
    oof_iso = np.zeros(n, dtype=np.float32)
    oof_risk = np.zeros(n, dtype=np.float32)
    oof_behavior = np.zeros((n, 3), dtype=np.float32)
    fold_id = np.full(n, -1, dtype=np.int8)
    models_xgb, models_lgb, models_cb = [], [], []
    models_dir, models_soft, models_iso, models_beh = [], [], [], []

    for fold, (tr, va) in enumerate(cv.split(X, behavior_y, groups), start=1):
        fold_id[va] = fold - 1
        mx = XGBClassifier(**xgb_params, random_state=SEED + fold)
        mx.fit(
            X.iloc[tr],
            y[tr],
            sample_weight=fit_weight[tr],
            eval_set=[(X.iloc[va], y[va])],
            sample_weight_eval_set=[stress_weight[va]],
            verbose=False,
        )
        oof_xgb[va] = mx.predict_proba(X.iloc[va])[:, 1]
        models_xgb.append(mx)

        ml = LGBMClassifier(**lgb_params, random_state=SEED + fold)
        ml.fit(
            X.iloc[tr],
            y[tr],
            sample_weight=fit_weight[tr],
            eval_set=[(X.iloc[va], y[va])],
            eval_sample_weight=[stress_weight[va]],
        )
        oof_lgb[va] = ml.predict_proba(X.iloc[va])[:, 1]
        models_lgb.append(ml)

        mc = CatBoostClassifier(**cb_params, random_seed=SEED + fold)
        mc.fit(
            X.iloc[tr],
            y[tr],
            sample_weight=fit_weight[tr],
            eval_set=[(X.iloc[va], y[va])],
            early_stopping_rounds=80,
            verbose=0,
        )
        oof_cb[va] = mc.predict_proba(X.iloc[va])[:, 1]
        models_cb.append(mc)

        for fam_id, store, bucket, pos_w in (
            (1, oof_dir, models_dir, 3.0),
            (2, oof_soft, models_soft, 3.0),
            (3, oof_iso, models_iso, 10.0),
        ):
            yk = (behavior_y == fam_id).astype(np.int8)
            wk = np.where(behavior_y == fam_id, pos_w, np.where(known, 1.0, 0.35))
            det = LGBMClassifier(
                n_estimators=900,
                learning_rate=0.03,
                num_leaves=35,
                min_child_samples=12,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_lambda=6.0,
                objective="binary",
                random_state=SEED + fold * 10 + fam_id,
                n_jobs=-1,
                verbose=-1,
            )
            det.fit(
                X.iloc[tr],
                yk[tr],
                sample_weight=wk[tr],
                eval_set=[(X.iloc[va], yk[va])],
                eval_sample_weight=[stress_weight[va]],
            )
            store[va] = det.predict_proba(X.iloc[va])[:, 1]
            bucket.append(det)

        oof_risk[va] = 0.40 * oof_xgb[va] + 0.35 * oof_lgb[va] + 0.25 * oof_cb[va]
        pos_tr, pos_va = tr[y[tr] == 1], va[y[va] == 1]
        counts = np.bincount(behavior_y[pos_tr] - 1, minlength=3)
        class_w = len(pos_tr) / (3.0 * np.maximum(counts, 1))
        b_params = dict(behavior_params)
        if len(pos_va) == 0:
            b_params.pop("early_stopping_rounds", None)
        bm = XGBClassifier(**b_params, random_state=SEED + fold)
        fit_kw = dict(sample_weight=class_w[behavior_y[pos_tr] - 1], verbose=False)
        if len(pos_va):
            fit_kw["eval_set"] = [(X.iloc[pos_va], behavior_y[pos_va] - 1)]
        bm.fit(X.iloc[pos_tr], behavior_y[pos_tr] - 1, **fit_kw)
        oof_behavior[va] = bm.predict_proba(X.iloc[va])
        models_beh.append(bm)
        lab_va = va[known[va]]
        print(
            f"Fold {fold}: labeled AP {average_precision_score(y[lab_va], oof_risk[lab_va]):.4f} | "
            f"PU-stress AP {average_precision_score(y[va], oof_risk[va], sample_weight=stress_weight[va]):.4f}"
        )

    family_oof = np.column_stack([oof_dir, oof_soft, oof_iso]).argmax(1)

    def behavior_map(active, row_mask, weights=None):
        w = None if weights is None else weights[row_mask]
        scores = []
        for k in range(3):
            class_score = np.where(active[row_mask] & (family_oof[row_mask] == k), oof_risk[row_mask], 0.0)
            scores.append(average_precision_score(behavior_y[row_mask] == k + 1, class_score, sample_weight=w))
        return float(np.mean(scores)), scores

    rate_grid = np.array([0.0050, 0.0065, 0.0080, 0.0095, 0.0110, 0.0125, 0.0140])
    rate_scores = np.array(
        [behavior_map(top_mask(oof_risk, r), np.ones(len(y), dtype=bool), stress_weight)[0] for r in rate_grid]
    )
    best_rate_score = float(rate_scores.max())
    behavior_positive_rate = float(rate_grid[rate_scores >= best_rate_score - 0.005].min())
    behavior_active_oof = top_mask(oof_risk, behavior_positive_rate)
    pair_ap_labeled = average_precision_score(y[known], oof_risk[known])
    pair_ap_stress = average_precision_score(y, oof_risk, sample_weight=stress_weight)
    behavior_map_labeled, _ = behavior_map(behavior_active_oof, known)
    behavior_map_stress, class_ap = behavior_map(behavior_active_oof, np.ones(len(y), dtype=bool), stress_weight)
    print(f"Labeled Pair AP {pair_ap_labeled:.4f} | PU-stress {pair_ap_stress:.4f}")
    print(f"Behavior MAP labeled {behavior_map_labeled:.4f} | PU-stress {behavior_map_stress:.4f}")
    print(f"Selected positive rate {behavior_positive_rate:.3%}")
    for name, score in zip(BEHAVIORS, class_ap):
        print(f"  {name}: {score:.4f}")

    print("Training evidence dual-engine...")
    positive_info = (
        labels.filter(pl.col("behavior_family").is_in(BEHAVIORS))
        .select(["pair_id", "behavior_family"])
        .with_columns(
            pl.when(pl.col("behavior_family") == "directed_transfer")
            .then(1)
            .when(pl.col("behavior_family") == "soft_play")
            .then(2)
            .otherwise(3)
            .cast(pl.Int8)
            .alias("behavior_id_true")
        )
    )
    evidence_keys = evidence.select(["pair_id", "hand_id"]).unique()
    pair_fold = pl.DataFrame(
        {"pair_id": train_df["pair_id"], "fold": fold_id, "behavior_id_pred": (family_oof + 1).astype(np.int8)}
    )
    evidence_base = (
        pl.scan_parquet(paths["dev_hand"])
        .select(["pair_id", "hand_id"] + hand_features)
        .join(positive_info.lazy(), on="pair_id")
        .join(evidence_keys.lazy().with_columns(pl.lit(1).alias("is_evidence")), on=["pair_id", "hand_id"], how="left")
        .with_columns(pl.col("is_evidence").fill_null(0).cast(pl.Int8))
        .collect()
        .join(pair_fold, on="pair_id")
    )
    rank_base = [c for c in hand_features if c not in skip]
    evidence_base = evidence_base.with_columns(
        [
            (pl.col(c).rank(method="average").over("pair_id") / pl.len().over("pair_id")).cast(pl.Float32).alias(
                f"{c}_pair_pct"
            )
            for c in rank_base
        ]
        + [
            (pl.col(c) / (pl.col(c).max().over("pair_id") + 1e-4)).cast(pl.Float32).alias(f"{c}_to_max")
            for c in rank_base
        ]
    ).sort(["pair_id", "hand_id"]).with_row_index("_row")
    ev_cols = hand_features + [f"{c}_pair_pct" for c in rank_base] + [f"{c}_to_max" for c in rank_base]
    X_ev = evidence_base.select(ev_cols).to_pandas().fillna(0).astype(np.float32)
    y_ev = evidence_base["is_evidence"].to_numpy().astype(np.int32)
    oof_glob = np.zeros(len(evidence_base), dtype=np.float32)
    oof_spec = np.zeros(len(evidence_base), dtype=np.float32)
    truth = {p: set(evidence.filter(pl.col("pair_id") == p)["hand_id"].to_list()) for p in positive_info["pair_id"]}

    for fold in range(5):
        tr_mask = (evidence_base["fold"] != fold).to_numpy()
        va_mask = (evidence_base["fold"] == fold).to_numpy()
        glob = HistGradientBoostingClassifier(
            max_iter=240,
            learning_rate=0.045,
            max_leaf_nodes=40,
            min_samples_leaf=20,
            l2_regularization=2.0,
            random_state=SEED + fold,
        )
        glob.fit(X_ev.iloc[tr_mask], y_ev[tr_mask])
        oof_glob[va_mask] = glob.predict_proba(X_ev.iloc[va_mask])[:, 1]
        for fam_id in (1, 2, 3):
            tr_part = evidence_base.filter((pl.col("fold") != fold) & (pl.col("behavior_id_true") == fam_id)).sort(
                "pair_id"
            )
            va_part = evidence_base.filter((pl.col("fold") == fold) & (pl.col("behavior_id_pred") == fam_id)).sort(
                "pair_id"
            )
            if tr_part.height == 0 or va_part.height == 0:
                continue
            groups_tr = tr_part.group_by("pair_id", maintain_order=True).len()["len"].to_numpy()
            X_tr = tr_part.select(ev_cols).to_pandas().fillna(0).astype(np.float32)
            y_tr = tr_part["is_evidence"].to_numpy()
            X_va = va_part.select(ev_cols).to_pandas().fillna(0).astype(np.float32)
            models = fit_family_quad(X_tr, y_tr, groups_tr, SEED + fold)
            oof_spec[va_part["_row"].to_numpy()] = predict_quad(models, X_va)

    evidence_oof = 0.60 * oof_spec + 0.40 * oof_glob
    pids = evidence_base["pair_id"].to_numpy()
    hids = evidence_base["hand_id"].to_numpy()
    print(f"Evidence OOF MAP@5 global {official_evidence_map5(pids, hids, oof_glob, truth):.4f}")
    print(f"Evidence OOF MAP@5 spec   {official_evidence_map5(pids, hids, oof_spec, truth):.4f}")
    print(f"Evidence OOF MAP@5 blend  {official_evidence_map5(pids, hids, evidence_oof, truth):.4f} (0.60 spec + 0.40 global, no z-blend)")

    ranker_global = HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.045,
        max_leaf_nodes=40,
        min_samples_leaf=20,
        l2_regularization=2.0,
        random_state=SEED,
    )
    ranker_global.fit(X_ev, y_ev)
    specialized = {}
    for fam_id in (1, 2, 3):
        fam_part = evidence_base.filter(pl.col("behavior_id_true") == fam_id).sort("pair_id")
        g = fam_part.group_by("pair_id", maintain_order=True).len()["len"].to_numpy()
        Xf = fam_part.select(ev_cols).to_pandas().fillna(0).astype(np.float32)
        yf = fam_part["is_evidence"].to_numpy()
        specialized[fam_id] = fit_family_quad(Xf, yf, g, SEED + fam_id)

    print("Scoring evaluation pairs...")
    eval_features = aggregate_pair_features(
        paths["eval_hand"],
        eval_pairs_prep,
        players,
        paths["baseline"],
        pair_agg_features,
        score_features,
        tail_features,
        score_q95,
    )
    eval_features = add_partner_field_contrasts(eval_features, seats, hands, paths["player"], "evaluation")
    missing = [c for c in feature_cols if c not in eval_features.columns]
    if missing:
        eval_features = eval_features.with_columns([pl.lit(0.0).cast(pl.Float32).alias(c) for c in missing])
    X_eval = matrix_from(eval_features, feature_cols)
    eval_risk = (
        0.40 * np.mean([m.predict_proba(X_eval)[:, 1] for m in models_xgb], axis=0)
        + 0.35 * np.mean([m.predict_proba(X_eval)[:, 1] for m in models_lgb], axis=0)
        + 0.25 * np.mean([m.predict_proba(X_eval)[:, 1] for m in models_cb], axis=0)
    )
    eval_family = np.column_stack(
        [
            np.mean([m.predict_proba(X_eval)[:, 1] for m in models_dir], axis=0),
            np.mean([m.predict_proba(X_eval)[:, 1] for m in models_soft], axis=0),
            np.mean([m.predict_proba(X_eval)[:, 1] for m in models_iso], axis=0),
        ]
    ).argmax(1)
    n_pos = max(1, int(round(len(eval_risk) * behavior_positive_rate)))
    active = np.zeros(len(eval_risk), dtype=bool)
    active[np.argsort(-eval_risk)[:n_pos]] = True
    predicted = np.where(active, np.array(BEHAVIORS)[eval_family], "none")
    eval_predictions = pl.DataFrame(
        {
            "pair_id": eval_features["pair_id"],
            "risk_score": eval_risk.astype(np.float32),
            "predicted_behavior": predicted.astype(str),
            "behavior_id_pred": (eval_family + 1).astype(np.int8),
        }
    )
    print(eval_predictions.group_by("predicted_behavior").len().sort("len", descending=True))

    behavior_map_df = eval_predictions.select(["pair_id", "behavior_id_pred"])
    eval_pair_ids = eval_predictions["pair_id"].to_list()
    dcol = "directed_priority" if "directed_priority" in hand_features else "directed_signal"
    scol = "soft_priority" if "soft_priority" in hand_features else "soft_signal"
    icol = "isolation_priority" if "isolation_priority" in hand_features else "isolation_signal"
    heuristic_top = (
        pl.scan_parquet(paths["eval_hand"])
        .select(["pair_id", "hand_id", dcol, scol, icol])
        .join(behavior_map_df.lazy(), on="pair_id")
        .with_columns(
            pl.when(pl.col("behavior_id_pred") == 1)
            .then(pl.col(dcol))
            .when(pl.col("behavior_id_pred") == 2)
            .then(pl.col(scol))
            .otherwise(pl.col(icol))
            .fill_null(0.0)
            .alias("_score")
        )
        .group_by("pair_id")
        .agg(pl.col("hand_id").sort_by("_score", descending=True).head(5).alias("hands"))
        .collect(engine="streaming")
    )
    rank_exprs = [
        (pl.col(c).rank(method="average").over("pair_id") / pl.len().over("pair_id")).cast(pl.Float32).alias(
            f"{c}_pair_pct"
        )
        for c in rank_base
    ] + [(pl.col(c) / (pl.col(c).max().over("pair_id") + 1e-4)).cast(pl.Float32).alias(f"{c}_to_max") for c in rank_base]
    print(f"Scoring evidence for {len(eval_pair_ids):,} pairs in chunks of {EVIDENCE_PAIR_CHUNK:,}...")
    model_parts: list[pl.DataFrame] = []
    for i in range(0, len(eval_pair_ids), EVIDENCE_PAIR_CHUNK):
        chunk_ids = eval_pair_ids[i : i + EVIDENCE_PAIR_CHUNK]
        chunk_hands = (
            pl.scan_parquet(paths["eval_hand"])
            .filter(pl.col("pair_id").is_in(chunk_ids))
            .join(behavior_map_df.lazy(), on="pair_id")
            .with_columns(rank_exprs)
            .select(["pair_id", "hand_id", "behavior_id_pred", *ev_cols])
            .sort(["pair_id", "hand_id"])
            .collect()
        )
        missing_ev = [c for c in ev_cols if c not in chunk_hands.columns]
        if missing_ev:
            chunk_hands = chunk_hands.with_columns([pl.lit(0.0).cast(pl.Float32).alias(c) for c in missing_ev])
        scores = np.empty(len(chunk_hands), dtype=np.float32)
        batch = 300_000
        for start in range(0, len(chunk_hands), batch):
            end = min(start + batch, len(chunk_hands))
            part = chunk_hands.slice(start, end - start)
            Xc = part.select(ev_cols).to_pandas().fillna(0).astype(np.float32)
            p_glob = ranker_global.predict_proba(Xc)[:, 1]
            p_dir = predict_quad(specialized[1], Xc)
            p_soft = predict_quad(specialized[2], Xc)
            p_iso = predict_quad(specialized[3], Xc)
            fam = part["behavior_id_pred"].to_numpy()
            p_spec = np.where(fam == 1, p_dir, np.where(fam == 2, p_soft, p_iso))
            scores[start:end] = (0.60 * p_spec + 0.40 * p_glob).astype(np.float32)
        model_parts.append(
            chunk_hands.select(["pair_id", "hand_id"])
            .with_columns(pl.Series("_score", scores))
            .group_by("pair_id")
            .agg(pl.col("hand_id").sort_by("_score", descending=True).head(5).alias("model_hands"))
        )
        print(f"  evidence pairs {i + 1:,}-{min(i + EVIDENCE_PAIR_CHUNK, len(eval_pair_ids)):,} / {len(eval_pair_ids):,}")
        del chunk_hands, scores
        gc.collect()
    model_top = pl.concat(model_parts) if model_parts else pl.DataFrame({"pair_id": [], "model_hands": []})
    evidence_wide = (
        heuristic_top.join(model_top, on="pair_id", how="left")
        .with_columns(pl.coalesce(["model_hands", "hands"]).alias("hands"))
        .drop("model_hands")
        .with_columns(
            [
                pl.col("hands").list.get(i, null_on_oob=True).fill_null("NO_EVIDENCE").alias(f"evidence_hand_{i + 1}")
                for i in range(5)
            ]
        )
        .drop("hands")
    )
    submission = (
        sample_sub.select("pair_id")
        .join(eval_predictions.select(["pair_id", "risk_score", "predicted_behavior"]), on="pair_id", how="left")
        .join(evidence_wide, on="pair_id", how="left")
        .with_columns([pl.col(f"evidence_hand_{i}").fill_null("NO_EVIDENCE") for i in range(1, 6)])
        .select(sample_sub.columns)
    )
    pdf = submission.to_pandas()
    assert list(pdf.columns) == list(sample_sub.columns)
    assert len(pdf) == len(eval_pairs)
    assert pdf["risk_score"].between(0, 1).all()
    assert not pdf.isna().any().any()
    out = OUTPUT_DIR / "submission.csv"
    submission.write_csv(out)
    print(f"Saved {out} | rows={len(submission):,}")
    print(submission.group_by("predicted_behavior").len().sort("len", descending=True))
    print(
        f"Projected (PU-stress, optimistic): "
        f"{0.70 * pair_ap_stress + 0.20 * official_evidence_map5(pids, hids, evidence_oof, truth) + 0.10 * behavior_map_stress:.4f}"
    )
    print(submission.head())


### 8.7 Run the pipeline

Writes `/kaggle/working/submission.csv`. First run ~hours on CPU (feature cache + 5-fold + evidence on 112k pairs). Later cache hits are mostly train/infer.


In [ ]:
main()
